In [ ]:
# BLOQUE 1 — Instala PostgreSQL, HypoPG y las librerías de Python
# Librerías de Python
!pip -q install psycopg2-binary lightgbm scikit-learn >/dev/null 2>&1
# PostgreSQL
!apt-get -qq -y install postgresql postgresql-contrib >/dev/null 2>&1
# HypoPG (índices hipotéticos, se usa solo en la base gemela de la V3)
!apt-get -qq -y install postgresql-$(ls /usr/lib/postgresql | sort -n | tail -1)-hypopg >/dev/null 2>&1
# Arranca PostgreSQL y le pone clave al usuario postgres
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';" >/dev/null 2>&1
print("PostgreSQL listo.")

In [ ]:
# BLOQUE 2 — Configuración general y conexión a PostgreSQL
import json, random, itertools, io, os, re, time, warnings
import numpy as np, pandas as pd
import psycopg2
import matplotlib.pyplot as plt
from statistics import median
warnings.filterwarnings("ignore")

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA)

ESCALA        = 1.0    # tamaño de las bases (1.0 = la tabla más grande con 150 a 400 mil filas)
RONDAS        = 30     # lanzamientos al azar: cada ronda elige una base y genera consultas
POR_RONDA     = 10     # consultas que se generan en cada ronda (15 x 10 = 150 consultas)
MAX_ORDENES   = 6      # cuántos órdenes de JOIN distintos se prueban por consulta
CALENTAMIENTO = 1      # ejecuciones que no se miden (llenan la caché de PostgreSQL)
REPETICIONES  = 3      # mediciones por plan; se usa la mediana para quitar ruido
LIMITE_MS     = 5000   # si un plan tarda más que esto se corta y se descarta

CONEXION = dict(host="localhost", dbname="postgres", user="postgres", password="postgres")

def conectar(base=None, solo_lectura=False):
    datos_conexion = dict(CONEXION)
    if base:
        datos_conexion["dbname"] = base
    if solo_lectura:
        datos_conexion["options"] = "-c default_transaction_read_only=on"
    c = psycopg2.connect(**datos_conexion)
    c.autocommit = True
    return c

TIEMPOS = {}

def avance(i, total, inicio, texto=""):
    paso = max(1, total // 10)
    if i == total or i % paso == 0:
        transcurrido = time.time() - inicio
        falta = transcurrido / max(i, 1) * (total - i)
        print(f"   {texto} {i}/{total} | {transcurrido/60:5.1f} min transcurridos | faltan ~{falta/60:4.1f} min", flush=True)

def fin_bloque(nombre, inicio):
    TIEMPOS[nombre] = time.time() - inicio
    print(f"[{nombre}] tardó {TIEMPOS[nombre]/60:.1f} min")

conexion_admin = conectar()
print("Conectado a PostgreSQL:", conexion_admin.server_version)

In [ ]:
# BLOQUE 3 — Crea las 5 bases de datos y las carga en PostgreSQL
def nom(prefijo):            return {"tipo": "nombre", "prefijo": prefijo}
def cat(k, sesgo=0.0):       return {"tipo": "categoria", "valores": k, "sesgo": sesgo}
def txt(valores, sesgo=0.0): return {"tipo": "texto", "valores": valores, "sesgo": sesgo}
def ent(a, b):               return {"tipo": "entero", "min": a, "max": b}
def num(a, b):               return {"tipo": "numero", "min": a, "max": b}
def fecha(desde, dias):      return {"tipo": "fecha", "desde": desde, "dias": dias}
def tabla(filas, pk, columnas, fija=False): return {"filas": filas, "pk": pk, "columnas": columnas, "fija": fija}
def rel(hija, columna, madre, sesgo=0.0, agrupada=False):
    return {"hija": hija, "columna": columna, "madre": madre, "sesgo": sesgo, "agrupada": agrupada}

REGIONES = ["AFRICA", "AMERICA", "ASIA", "EUROPE", "MIDDLE EAST"]
NACIONES = ["ALGERIA", "ARGENTINA", "BRAZIL", "CANADA", "EGYPT", "ETHIOPIA", "FRANCE", "GERMANY", "INDIA",
            "INDONESIA", "IRAN", "IRAQ", "JAPAN", "JORDAN", "KENYA", "MOROCCO", "MOZAMBIQUE", "PERU", "CHINA",
            "ROMANIA", "SAUDI ARABIA", "VIETNAM", "RUSSIA", "UNITED KINGDOM", "UNITED STATES"]

BASES_ORIGINALES = {
 "ventas_tpch": dict(
   tablas={
    "region":   tabla(5, "r_id", {"r_nombre": txt(REGIONES)}, fija=True),
    "nacion":   tabla(25, "n_id", {"n_nombre": txt(NACIONES)}, fija=True),
    "proveedor": tabla(1000, "s_id", {"s_nombre": nom("Sup"), "s_saldo": num(-1000, 10000)}),
    "cliente":  tabla(15000, "c_id", {"c_nombre": nom("Cli"), "c_segmento": txt(["AUTOMOBILE", "BUILDING", "FURNITURE", "MACHINERY", "HOUSEHOLD"], .3),
                                      "c_saldo": num(-500, 9999)}),
    "parte":    tabla(10000, "p_id", {"p_nombre": nom("Parte"), "p_marca": txt([f"Brand#{m}{n}" for m in range(1, 6) for n in range(1, 6)], .3),
                                      "p_tamano": ent(1, 50)}),
    "parte_proveedor": tabla(40000, "ps_id", {"ps_costo": num(1, 1000), "ps_stock": ent(1, 9999)}),
    "orden":    tabla(60000, "o_id", {"o_fecha": fecha("1995-01-01", 1460), "o_total": num(10, 90000),
                                      "o_prioridad": txt(["1-URGENT", "2-HIGH", "3-MEDIUM", "4-NOT SPECIFIED", "5-LOW"], .4)}),
    "linea":    tabla(150000, "l_id", {"l_cantidad": ent(1, 50), "l_precio": num(100, 80000), "l_descuento": num(0, 0.1)}),
   },
   relaciones=[rel("nacion", "n_region", "region"), rel("proveedor", "s_nacion", "nacion", .6),
               rel("cliente", "c_nacion", "nacion", .6), rel("parte_proveedor", "ps_parte", "parte", agrupada=True),
               rel("parte_proveedor", "ps_proveedor", "proveedor"), rel("orden", "o_cliente", "cliente", .3),
               rel("linea", "l_orden", "orden", agrupada=True), rel("linea", "l_parte", "parte", .5),
               rel("linea", "l_proveedor", "proveedor")]),
 "tienda_online": dict(
   tablas={
    "pais":      tabla(30, "pa_id", {"pa_nombre": nom("Pais"), "pa_continente": cat(6, .5)}, fija=True),
    "ciudad":    tabla(600, "ci_id", {"ci_nombre": nom("Ciudad"), "ci_poblacion": ent(1000, 5000000)}, fija=True),
    "categoria": tabla(60, "ca_id", {"ca_nombre": nom("Cat"), "ca_tipo": txt(["ropa", "hogar", "tecnologia", "deporte", "libros", "juguetes"], .4)}, fija=True),
    "vendedor":  tabla(2000, "ve_id", {"ve_nombre": nom("Vend"), "ve_rating": num(1, 5)}),
    "cliente":   tabla(30000, "cl_id", {"cl_nombre": nom("Cli"), "cl_segmento": txt(["nuevo", "frecuente", "vip", "inactivo"], .5),
                                        "cl_alta": fecha("2018-01-01", 2190)}),
    "producto":  tabla(20000, "pr_id", {"pr_nombre": nom("Prod"), "pr_precio": num(1, 2000), "pr_stock": ent(0, 500)}),
    "pedido":    tabla(100000, "pe_id", {"pe_fecha": fecha("2020-01-01", 1825), "pe_total": num(5, 5000),
                                         "pe_estado": txt(["entregado", "enviado", "pendiente", "cancelado", "devuelto"], .7)}),
    "detalle":   tabla(250000, "de_id", {"de_cantidad": ent(1, 10), "de_precio": num(1, 2000)}),
    "pago":      tabla(100000, "pg_id", {"pg_metodo": txt(["tarjeta", "transferencia", "efectivo", "billetera"], .5), "pg_monto": num(5, 5000)}),
    "envio":     tabla(90000, "en_id", {"en_dias": ent(1, 30), "en_costo": num(0, 50)}),
    "resena":    tabla(60000, "re_id", {"re_nota": ent(1, 5), "re_fecha": fecha("2020-01-01", 1825)}),
   },
   relaciones=[rel("ciudad", "ci_pais", "pais", .5), rel("vendedor", "ve_ciudad", "ciudad", .5),
               rel("cliente", "cl_ciudad", "ciudad", .6), rel("producto", "pr_categoria", "categoria", .4),
               rel("producto", "pr_vendedor", "vendedor", .5), rel("pedido", "pe_cliente", "cliente", .3),
               rel("detalle", "de_pedido", "pedido", agrupada=True), rel("detalle", "de_producto", "producto", .6),
               rel("pago", "pg_pedido", "pedido", agrupada=True), rel("envio", "en_pedido", "pedido", agrupada=True),
               rel("envio", "en_ciudad", "ciudad", .5), rel("resena", "re_producto", "producto", .7),
               rel("resena", "re_cliente", "cliente")]),
 "universidad": dict(
   tablas={
    "facultad":    tabla(12, "fa_id", {"fa_nombre": nom("Fac"), "fa_campus": cat(3, .5)}, fija=True),
    "carrera":     tabla(80, "ca_id", {"ca_nombre": nom("Carrera"), "ca_duracion": ent(8, 12)}, fija=True),
    "profesor":    tabla(1500, "pf_id", {"pf_nombre": nom("Prof"), "pf_grado": txt(["magister", "doctor", "licenciado"], .5),
                                         "pf_ingreso": fecha("1990-01-01", 12000)}),
    "estudiante":  tabla(40000, "es_id", {"es_nombre": nom("Est"), "es_ingreso": fecha("2015-01-01", 3650), "es_region": cat(16, .6),
                                          "es_beca": txt(["ninguna", "parcial", "completa"], .6)}),
    "curso":       tabla(3000, "cu_id", {"cu_nombre": nom("Curso"), "cu_creditos": ent(2, 10), "cu_nivel": ent(1, 10)}),
    "seccion":     tabla(12000, "se_id", {"se_semestre": cat(20, .3), "se_cupos": ent(10, 120)}),
    "inscripcion": tabla(300000, "in_id", {"in_estado": txt(["aprobado", "reprobado", "retirado", "cursando"], .6),
                                           "in_fecha": fecha("2015-01-01", 3650)}),
    "evaluacion":  tabla(48000, "ev_id", {"ev_tipo": txt(["certamen", "tarea", "laboratorio", "examen", "proyecto"], .4), "ev_peso": ent(5, 40)}),
    "nota":        tabla(300000, "no_id", {"no_valor": num(1, 7)}),
   },
   relaciones=[rel("carrera", "ca_facultad", "facultad", .4), rel("profesor", "pf_facultad", "facultad", .4),
               rel("estudiante", "es_carrera", "carrera", .5), rel("curso", "cu_carrera", "carrera", .3),
               rel("seccion", "se_curso", "curso", .3), rel("seccion", "se_profesor", "profesor", .4),
               rel("inscripcion", "in_estudiante", "estudiante", agrupada=True), rel("inscripcion", "in_seccion", "seccion", .5),
               rel("evaluacion", "ev_seccion", "seccion", agrupada=True), rel("nota", "no_evaluacion", "evaluacion", .3),
               rel("nota", "no_estudiante", "estudiante")]),
 "hospital": dict(
   tablas={
    "region":       tabla(16, "rg_id", {"rg_nombre": nom("Region"), "rg_zona": cat(3, .5)}, fija=True),
    "hospital":     tabla(120, "ho_id", {"ho_nombre": nom("Hosp"), "ho_camas": ent(20, 900), "ho_tipo": txt(["publico", "privado"], .6)}, fija=True),
    "especialidad": tabla(40, "sp_id", {"sp_nombre": nom("Esp"), "sp_area": cat(5, .4)}, fija=True),
    "medico":       tabla(5000, "me_id", {"me_nombre": nom("Med"), "me_ingreso": fecha("1985-01-01", 14000)}),
    "paciente":     tabla(80000, "pa_id", {"pa_nombre": nom("Pac"), "pa_nacimiento": fecha("1940-01-01", 30000),
                                           "pa_prevision": txt(["fonasa_a", "fonasa_b", "fonasa_c", "fonasa_d", "isapre", "particular"], .6)}),
    "atencion":     tabla(250000, "at_id", {"at_fecha": fecha("2019-01-01", 2190), "at_costo": num(0, 2000000),
                                            "at_tipo": txt(["consulta", "urgencia", "control", "hospitalizacion", "telemedicina"], .6)}),
    "codigo_diagnostico": tabla(2000, "cd_id", {"cd_nombre": nom("CIE"), "cd_capitulo": cat(22, .4)}),
    "diagnostico":  tabla(250000, "di_id", {"di_principal": txt(["si", "no"], .5)}),
    "medicamento":  tabla(3000, "md_id", {"md_nombre": nom("Med"), "md_precio": num(500, 90000),
                                          "md_tipo": txt(["analgesico", "antibiotico", "antihipertensivo", "antidiabetico", "otro"], .5)}),
    "receta":       tabla(200000, "rc_id", {"rc_dosis": ent(1, 4), "rc_dias": ent(1, 90)}),
   },
   relaciones=[rel("hospital", "ho_region", "region", .5), rel("medico", "me_hospital", "hospital", .4),
               rel("medico", "me_especialidad", "especialidad", .3), rel("paciente", "pa_region", "region", .6),
               rel("atencion", "at_paciente", "paciente", .3), rel("atencion", "at_medico", "medico", .5),
               rel("diagnostico", "di_atencion", "atencion", agrupada=True), rel("diagnostico", "di_codigo", "codigo_diagnostico", .7),
               rel("receta", "rc_atencion", "atencion", agrupada=True), rel("receta", "rc_medicamento", "medicamento", .6)]),
 "cine_streaming": dict(
   tablas={
    "pais":         tabla(60, "pa_id", {"pa_nombre": nom("Pais"), "pa_continente": cat(6, .5)}, fija=True),
    "compania":     tabla(8000, "co_id", {"co_nombre": nom("Comp"), "co_tipo": txt(["productora", "distribuidora", "efectos"], .6)}),
    "genero":       tabla(25, "ge_id", {"ge_nombre": nom("Genero")}, fija=True),
    "pelicula":     tabla(60000, "pe_id", {"pe_titulo": nom("Peli"), "pe_anio": ent(1920, 2025), "pe_duracion": ent(5, 240),
                                           "pe_tipo": txt(["pelicula", "serie", "corto", "documental", "episodio"], .6)}),
    "persona":      tabla(150000, "pr_id", {"pr_nombre": nom("Pers"), "pr_genero": txt(["m", "f", "x"], .5),
                                            "pr_nacimiento": fecha("1900-01-01", 40000)}),
    "reparto":      tabla(400000, "rp_id", {"rp_rol": txt(["actor", "actriz", "director", "guionista", "productor", "musica"], .7),
                                            "rp_orden": ent(1, 50)}),
    "pelicula_genero": tabla(90000, "pg_id", {}),
    "produccion":   tabla(70000, "pc_id", {"pc_presupuesto": num(1000, 300000000)}),
    "usuario":      tabla(100000, "us_id", {"us_nombre": nom("User"), "us_plan": txt(["basico", "estandar", "premium"], .5),
                                            "us_alta": fecha("2015-01-01", 3650)}),
    "calificacion": tabla(300000, "cf_id", {"cf_nota": ent(1, 10), "cf_fecha": fecha("2015-01-01", 3650)}),
   },
   relaciones=[rel("compania", "co_pais", "pais", .6), rel("persona", "pr_pais", "pais", .5),
               rel("usuario", "us_pais", "pais", .6), rel("reparto", "rp_pelicula", "pelicula", .5),
               rel("reparto", "rp_persona", "persona", .4), rel("pelicula_genero", "pg_pelicula", "pelicula", agrupada=True),
               rel("pelicula_genero", "pg_genero", "genero", .5), rel("produccion", "pc_pelicula", "pelicula"),
               rel("produccion", "pc_compania", "compania", .6), rel("calificacion", "cf_pelicula", "pelicula", .7),
               rel("calificacion", "cf_usuario", "usuario", .3)]),
}

os.makedirs("bases", exist_ok=True)
for nombre, spec in BASES_ORIGINALES.items():
    with open(f"bases/{nombre}.json", "w") as f:
        json.dump({"nombre": nombre, **spec}, f, indent=1)
ARCHIVOS = sorted(f"bases/{a}" for a in os.listdir("bases") if a.endswith(".json"))
print("Archivos de bases de datos:", ARCHIVOS)

def filas_de(t):
    return t["filas"] if t.get("fija") else max(10, int(t["filas"] * ESCALA))

def generar_columna(col, n, rng):
    tipo = col["tipo"]
    if tipo == "nombre":
        return np.array([f"{col['prefijo']}_{i}" for i in range(n)], dtype=object)
    if tipo in ("categoria", "texto"):
        k = col["valores"] if tipo == "categoria" else len(col["valores"])
        calientes = max(1, k // 8)   # los pocos valores que se repiten mucho
        idx = np.where(rng.random(n) < col.get("sesgo", 0), rng.integers(0, calientes, n), rng.integers(0, k, n))
        if col.get("sesgo", 0) > 0:
            populares = max(1, n // 20)
            idx[:populares] = rng.integers(0, calientes, populares)
        return idx if tipo == "categoria" else np.array(col["valores"], dtype=object)[idx]
    if tipo == "entero":
        return rng.integers(col["min"], col["max"] + 1, n)
    if tipo == "numero":
        return np.round(rng.uniform(col["min"], col["max"], n), 2)
    if tipo == "fecha":
        return (np.datetime64(col["desde"]) + rng.integers(0, col["dias"], n).astype("timedelta64[D]")).astype(str)

TIPO_SQL = {"nombre": "text", "categoria": "int", "texto": "text", "entero": "int", "numero": "numeric", "fecha": "date"}

def copiar(cur, nombre_tabla, df):
    buffer = io.StringIO()
    df.to_csv(buffer, sep="\t", header=False, index=False)
    buffer.seek(0)
    cur.copy_expert(f"COPY {nombre_tabla} ({','.join(df.columns)}) FROM STDIN", buffer)

def cargar_base(spec):
    rng = np.random.default_rng(SEMILLA)
    nombre_bd = f"optim_{spec['nombre']}"
    with conexion_admin.cursor() as cur:
        cur.execute(f"DROP DATABASE IF EXISTS {nombre_bd}")
        cur.execute(f"CREATE DATABASE {nombre_bd}")
    c = conectar(nombre_bd)
    filas = {t: filas_de(d) for t, d in spec["tablas"].items()}
    with c.cursor() as cur:
        for t, d in spec["tablas"].items():
            columnas = {d["pk"]: np.arange(filas[t])}
            for nombre_col, col in d["columnas"].items():
                columnas[nombre_col] = generar_columna(col, filas[t], rng)
            for r in spec["relaciones"]:
                if r["hija"] == t:
                    n_madre = filas[r["madre"]]
                    calientes = max(1, n_madre // 20)   # el 5% de las filas madre recibe la fracción "sesgo"
                    v = np.where(rng.random(filas[t]) < r["sesgo"], rng.integers(0, calientes, filas[t]),
                                 rng.integers(0, n_madre, filas[t]))
                    columnas[r["columna"]] = np.sort(v) if r["agrupada"] else v
            ddl = [f"{d['pk']} int PRIMARY KEY"] + [f"{k} {TIPO_SQL[v['tipo']]}" for k, v in d["columnas"].items()] \
                + [f"{r['columna']} int" for r in spec["relaciones"] if r["hija"] == t]
            cur.execute(f"CREATE TABLE {t} ({', '.join(ddl)})")
            copiar(cur, t, pd.DataFrame(columnas))
        for ddl in indices_base_de(spec):
            cur.execute(ddl)
        cur.execute("ANALYZE")
        cur.execute("SELECT pg_database_size(current_database())")
        mb = cur.fetchone()[0] / 2**20
    return c, dict(base=spec["nombre"], tablas=len(spec["tablas"]), relaciones=len(spec["relaciones"]),
                   filas=sum(filas.values()), tabla_mayor=max(filas.values()), mb=round(mb, 1))

def indices_base_de(spec):
    grandes = sorted(spec["tablas"], key=lambda t: -spec["tablas"][t]["filas"])[:3]
    return [f"CREATE INDEX ON {r['hija']}({r['columna']})" for r in spec["relaciones"] if r["hija"] in grandes]

inicio = time.time()
BASES, CONEXIONES, resumen_bases = {}, {}, []
for i, archivo in enumerate(ARCHIVOS, 1):
    with open(archivo) as f:
        spec = json.load(f)
    BASES[spec["nombre"]] = spec
    CONEXIONES[spec["nombre"]], info = cargar_base(spec)
    resumen_bases.append(info)
    avance(i, len(ARCHIVOS), inicio, f"cargada {spec['nombre']:15s}")
display(pd.DataFrame(resumen_bases))
fin_bloque("Bloque 3 - cargar las 5 bases", inicio)

In [ ]:
# BLOQUE 4 — Genera consultas al azar: cada ronda elige una de las 5 bases
def usar_base(nombre):
    global BASE_ACTIVA, conexion, ARISTAS, VECINOS, HIJAS, FILTROS, TABLAS, CLAVES_PRIMARIAS, INDICES_BASE, FILAS_TABLA
    spec = BASES[nombre]
    BASE_ACTIVA, conexion = nombre, CONEXIONES[nombre]
    TABLAS = list(spec["tablas"])
    FILAS_TABLA = {t: filas_de(d) for t, d in spec["tablas"].items()}
    CLAVES_PRIMARIAS = {d["pk"] for d in spec["tablas"].values()}
    INDICES_BASE = indices_base_de(spec)
    ARISTAS, HIJAS = {}, {}
    for r in spec["relaciones"]:
        HIJAS.setdefault(r["madre"], set()).add(r["hija"])
        ARISTAS[frozenset({r["hija"], r["madre"]})] = f"{r['hija']}.{r['columna']} = {r['madre']}.{spec['tablas'][r['madre']]['pk']}"
    VECINOS = {}
    for arista in ARISTAS:
        a, b = tuple(arista); VECINOS.setdefault(a, set()).add(b); VECINOS.setdefault(b, set()).add(a)
    FILTROS = {t: filtros_de(t, d) for t, d in spec["tablas"].items()}
    FILTROS = {t: f for t, f in FILTROS.items() if f}

def filtros_de(nombre_tabla, d):
    salida = []
    for c, col in d["columnas"].items():
        q, t = f"{nombre_tabla}.{c}", col["tipo"]
        if t == "categoria":
            salida += [f"{q} = 0", f"{q} = {col['valores'] // 2}", f"{q} = {col['valores'] - 1}"]
        elif t == "texto":
            salida += [f"{q} = '{v}'" for v in col["valores"][:6]]
        elif t in ("entero", "numero"):
            lo, hi = col["min"], col["max"]
            corte = (lambda p: int(round(lo + p * (hi - lo)))) if t == "entero" else (lambda p: round(lo + p * (hi - lo), 2))
            salida += [f"{q} > {corte(.5)}", f"{q} > {corte(.9)}", f"{q} > {corte(.98)}", f"{q} < {corte(.1)}"]
        elif t == "fecha":
            salida += [f"{q} >= DATE '{np.datetime64(col['desde']) + np.timedelta64(int(col['dias'] * p), 'D')}'" for p in (.3, .9)]
    return salida

def cruce_gigante(conjunto, v):
    for madre in VECINOS[v]:
        if madre in conjunto and v in HIJAS.get(madre, ()):
            otras = [h for h in HIJAS[madre] if h in conjunto]
            if otras and FILAS_TABLA[madre] < 0.2 * max(FILAS_TABLA[h] for h in otras + [v]):
                return True
    return False

def subconjunto_aleatorio(k):
    conjunto = [random.choices(TABLAS, weights=[FILAS_TABLA[t] for t in TABLAS])[0]]
    while len(conjunto) < k:
        vecinas = sorted(v for t in conjunto for v in VECINOS[t] if v not in conjunto and not cruce_gigante(conjunto, v))
        if not vecinas:
            break
        conjunto.append(random.choice(vecinas))
    return conjunto

def generar_consultas(n, vistas):
    salida, intentos = [], 0
    while len(salida) < n and intentos < n * 30:
        intentos += 1
        tablas = subconjunto_aleatorio(random.randint(3, 6))
        if len(tablas) < 3:
            continue
        filtros = [random.choice(FILTROS[t]) for t in tablas if t in FILTROS and random.random() < 0.6]
        clave = (BASE_ACTIVA, frozenset(tablas), tuple(sorted(filtros)))
        if clave in vistas:          # no repetir consultas
            continue
        vistas.add(clave); salida.append((tablas, filtros))
    return salida

def construir_sql(orden, filtros):
    sql = f"SELECT count(*) FROM {orden[0]}"; unidas = [orden[0]]
    for t in orden[1:]:
        condiciones = [ARISTAS[frozenset({t, x})] for x in unidas if frozenset({t, x}) in ARISTAS]
        sql += f" JOIN {t} ON {' AND '.join(condiciones)}"; unidas.append(t)
    if filtros:
        sql += " WHERE " + " AND ".join(filtros)
    return sql

def orden_valido(orden):
    return all(any(frozenset({orden[i], x}) in ARISTAS for x in orden[:i]) for i in range(1, len(orden)))

def ordenes_conectados(tablas, maximo, rng=random):
    resultado = []
    for permutacion in itertools.permutations(tablas):
        if orden_valido(permutacion):
            resultado.append(permutacion)
        if len(resultado) >= maximo * 6:
            break
    rng.shuffle(resultado)
    return resultado[:maximo]

def por_base(lista_consultas):
    grupos = {}
    for q in lista_consultas:
        grupos.setdefault(BASE_DE[q], []).append(q)
    return grupos.items()

inicio = time.time()
CONSULTAS, BASE_DE, vistas, lanzamientos = [], [], set(), []
for ronda in range(1, RONDAS + 1):
    archivo = random.choice(ARCHIVOS)                 # el lanzamiento: un archivo al azar
    nombre = os.path.basename(archivo)[:-5]
    usar_base(nombre)
    nuevas = generar_consultas(POR_RONDA, vistas)
    CONSULTAS += nuevas; BASE_DE += [nombre] * len(nuevas)
    lanzamientos.append(dict(ronda=ronda, archivo=archivo, consultas=len(nuevas)))
display(pd.DataFrame(lanzamientos))
mezcla = pd.Series(BASE_DE).value_counts()
print("Mezcla de consultas por base:", mezcla.to_dict())
tamanos = pd.Series([len(t) for t, _ in CONSULTAS]).value_counts().sort_index().to_dict()
print(f"Consultas generadas: {len(CONSULTAS)} | tablas por consulta: {tamanos}")
usar_base(BASE_DE[0])
print("Ejemplo:", construir_sql(*CONSULTAS[0]))
fin_bloque("Bloque 4 - lanzamientos y consultas", inicio)

In [ ]:
# BLOQUE 5 — Prueba varios planes por consulta y mide su tiempo real (el dataset)
def configurar_sesion(cur, forzar):
    if forzar:
        cur.execute("SET join_collapse_limit=1; SET from_collapse_limit=1; SET geqo=off;")
    else:
        cur.execute("SET join_collapse_limit TO DEFAULT; SET from_collapse_limit TO DEFAULT; SET geqo TO DEFAULT;")
    cur.execute(f"SET statement_timeout='{LIMITE_MS}';")

def plan_estimado(cur, sql):
    cur.execute("EXPLAIN (FORMAT JSON) " + sql)
    return cur.fetchone()[0][0]["Plan"]

def tiempo_real(cur, sql):
    for _ in range(CALENTAMIENTO):
        cur.execute("EXPLAIN (ANALYZE, FORMAT JSON) " + sql); cur.fetchone()
    tiempos = []
    for _ in range(REPETICIONES):
        cur.execute("EXPLAIN (ANALYZE, FORMAT JSON) " + sql)
        tiempos.append(cur.fetchone()[0][0]["Plan"]["Actual Total Time"])
    return median(tiempos)

NODOS = ["Seq Scan","Index Scan","Index Only Scan","Bitmap Heap Scan","Bitmap Index Scan",
         "Nested Loop","Hash Join","Merge Join","Hash","Sort","Aggregate","Materialize","Gather"]

def extraer_caracteristicas(plan, n_tablas):
    conteo = {f"n_{nt}": 0 for nt in NODOS}
    suma   = {"n_nodos":0, "prof_max":0, "filas_est_sum":0.0, "costo_sum":0.0, "ancho_sum":0.0, "n_scans":0}
    def recorrer(nodo, prof):
        tipo = nodo.get("Node Type", "")
        if tipo in NODOS: conteo[f"n_{tipo}"] += 1
        suma["n_nodos"] += 1
        suma["prof_max"] = max(suma["prof_max"], prof)
        suma["filas_est_sum"] += nodo.get("Plan Rows", 0) or 0
        suma["costo_sum"]     += nodo.get("Total Cost", 0) or 0
        suma["ancho_sum"]     += nodo.get("Plan Width", 0) or 0
        if "Scan" in tipo: suma["n_scans"] += 1
        for hijo in nodo.get("Plans", []): recorrer(hijo, prof + 1)
    recorrer(plan, 0)
    c = {**conteo, **suma}
    c["raiz_costo"]     = plan.get("Total Cost", 0)     # costo total estimado por PostgreSQL
    c["raiz_costo_ini"] = plan.get("Startup Cost", 0)   # costo antes de entregar la primera fila
    c["raiz_filas_est"] = plan.get("Plan Rows", 0)
    c["raiz_ancho"]     = plan.get("Plan Width", 0)
    c["n_tablas"]       = n_tablas
    return c

APAGAR = [("hashjoin",), ("mergejoin",), ("nestloop",),
          ("hashjoin", "mergejoin"), ("hashjoin", "nestloop"), ("mergejoin", "nestloop")]

registros = []
tiempos_nativos = {}   # consulta_id -> tiempo real del plan que elige PostgreSQL solo
inicio, hechas = time.time(), 0
for base, lista in por_base(range(len(CONSULTAS))):
    usar_base(base)
    with conexion.cursor() as cur:
        for consulta_id in lista:
            tablas, filtros = CONSULTAS[consulta_id]
            hechas += 1; avance(hechas, len(CONSULTAS), inicio, "consultas medidas")
            configurar_sesion(cur, forzar=False)
            sql_nativo = construir_sql(tablas, filtros)
            try:
                estimado = plan_estimado(cur, sql_nativo)
                tiempos_nativos[consulta_id] = tiempo_real(cur, sql_nativo)
            except Exception:
                continue                      # se pasó del límite de tiempo: se descarta la consulta
            fila = extraer_caracteristicas(estimado, len(tablas))
            fila.update(consulta_id=consulta_id, base=base, orden="(PostgreSQL decide)", tiempo_ms=tiempos_nativos[consulta_id])
            registros.append(fila)
            vistos = {json.dumps(estimado, sort_keys=True)}
            for apagados in APAGAR:
                for metodo in apagados:
                    cur.execute(f"SET enable_{metodo}=off;")
                try:
                    estimado = plan_estimado(cur, sql_nativo)
                    firma = json.dumps(estimado, sort_keys=True)
                    if firma not in vistos:          # si el plan no cambió, no se mide de nuevo
                        vistos.add(firma)
                        real = tiempo_real(cur, sql_nativo)
                        fila = extraer_caracteristicas(estimado, len(tablas))
                        fila.update(consulta_id=consulta_id, base=base, orden="(PostgreSQL, sin " + " ni ".join(apagados) + ")", tiempo_ms=real)
                        registros.append(fila)
                except Exception:
                    pass
                cur.execute("RESET enable_hashjoin; RESET enable_mergejoin; RESET enable_nestloop;")
            configurar_sesion(cur, forzar=True)
            for orden in ordenes_conectados(tablas, MAX_ORDENES):
                sql = construir_sql(orden, filtros)
                try:
                    estimado = plan_estimado(cur, sql)
                    real = tiempo_real(cur, sql)
                except Exception:
                    continue
                fila = extraer_caracteristicas(estimado, len(orden))
                fila.update(consulta_id=consulta_id, base=base, orden=" > ".join(orden), tiempo_ms=real)
                registros.append(fila)
        configurar_sesion(cur, forzar=False)

datos = pd.DataFrame(registros).fillna(0)
datos = datos[datos.groupby("consulta_id")["consulta_id"].transform("size") >= 2].reset_index(drop=True)
CARACTERISTICAS = [c for c in datos.columns if c not in ("consulta_id", "base", "orden", "tiempo_ms")]
print(f"Dataset: {len(datos)} planes de {datos.consulta_id.nunique()} consultas "
      f"x {len(CARACTERISTICAS)} características")
print("Planes por base:", datos.groupby("base").size().to_dict())
fin_bloque("Bloque 5 - capturar planes", inicio)

In [ ]:
# BLOQUE 6 — Un vistazo al dataset
print(f"Planes: {len(datos)} | consultas: {datos.consulta_id.nunique()} | "
      f"planes por consulta (promedio): {datos.groupby('consulta_id').size().mean():.1f}")
print("\nTiempo real por plan (ms):")
print(datos["tiempo_ms"].describe(percentiles=[.25, .5, .75, .95]).round(2).to_string())

por_consulta = datos.groupby("consulta_id")["tiempo_ms"].agg(["min", "max"])
por_consulta["peor/mejor"] = por_consulta["max"] / por_consulta["min"].clip(lower=1e-3)
print(f"\nEl peor orden es, en mediana, {por_consulta['peor/mejor'].median():.1f} veces más lento que el mejor")
print(f"Consultas donde el orden cambia el tiempo más de 2 veces: {(por_consulta['peor/mejor'] > 2).mean():.0%}")

spearman_costo = datos.groupby("consulta_id").apply(
    lambda g: g["raiz_costo"].corr(g["tiempo_ms"], method="spearman")).mean()
print(f"Correlación costo estimado vs tiempo real (dentro de cada consulta): {spearman_costo:.2f}")

por_consulta["base"] = [BASE_DE[q] for q in por_consulta.index]
print("\nPor base:")
print(por_consulta.groupby("base").agg(consultas=("min", "size"), mediana_peor_mejor=("peor/mejor", "median"),
                                       tiempo_mediano_ms=("min", "median")).round(2).to_string())

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(np.log10(datos["tiempo_ms"].clip(lower=0.01)), bins=40, color="#2563eb")
ax[0].set_xlabel("log10(tiempo real en ms)"); ax[0].set_ylabel("cantidad de planes")
ax[0].set_title("Distribución de los tiempos")
ax[1].hist(np.log10(por_consulta["peor/mejor"]), bins=30, color="#ea580c")
ax[1].set_xlabel("log10(peor orden / mejor orden)"); ax[1].set_ylabel("cantidad de consultas")
ax[1].set_title("Cuánto importa el orden de JOIN")
ax[2].scatter(datos["raiz_costo"], datos["tiempo_ms"], s=8, alpha=.4, color="#16a34a")
ax[2].set_xscale("log"); ax[2].set_yscale("log")
ax[2].set_xlabel("costo estimado por PostgreSQL"); ax[2].set_ylabel("tiempo real (ms)")
ax[2].set_title("Costo estimado vs tiempo real")
plt.tight_layout(); plt.show()

In [ ]:
# BLOQUE 7 — Separa los datos en entrenamiento, validación y prueba
inicio_bloque7 = time.time()
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ids = sorted(datos["consulta_id"].unique())
random.Random(SEMILLA).shuffle(ids)
n = len(ids)
ids_entrena = set(ids[:int(n * .6)])
ids_valida  = set(ids[int(n * .6):int(n * .8)])
ids_prueba  = set(ids[int(n * .8):])

entrena = datos[datos.consulta_id.isin(ids_entrena)]
valida  = datos[datos.consulta_id.isin(ids_valida)]
prueba  = datos[datos.consulta_id.isin(ids_prueba)]

def objetivo(df):
    return np.log1p(df["tiempo_ms"])

for nombre, parte in [("entrenamiento", entrena), ("validación", valida), ("prueba", prueba)]:
    print(f"{nombre:14s}: {parte.consulta_id.nunique():3d} consultas | {len(parte):4d} planes | "
          f"por base: {parte.drop_duplicates('consulta_id').base.value_counts().to_dict()}")

In [ ]:
# BLOQUE 7.2 — Validación cruzada
PARAMETROS_BASE = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                       subsample=0.9, subsample_freq=1, colsample_bytree=0.9,
                       random_state=SEMILLA, verbosity=-1)

def evaluar_eleccion(df, puntaje):
    df = df.assign(puntaje=np.asarray(puntaje))
    aciertos, sobrecostos, tiempo_elegido, tiempo_mejor = 0, [], 0.0, 0.0
    for _, g in df.groupby("consulta_id"):
        elegido = g.loc[g["puntaje"].idxmin(), "tiempo_ms"]
        mejor = g["tiempo_ms"].min()
        aciertos += int(elegido <= mejor * 1.05)   # se cuenta acierto si queda a menos de 5% del mejor
        sobrecostos.append(elegido / max(mejor, 1e-3) - 1)
        tiempo_elegido += elegido; tiempo_mejor += mejor
    return dict(top1=aciertos / df.consulta_id.nunique(), sobrecosto=float(np.median(sobrecostos)),
                tiempo_elegido=tiempo_elegido, tiempo_mejor=tiempo_mejor)

def metricas(df, pred_log):
    pred_ms = np.expm1(pred_log)
    e = evaluar_eleccion(df, pred_log)
    return dict(MAE=mean_absolute_error(df["tiempo_ms"], pred_ms),
                RMSE=mean_squared_error(df["tiempo_ms"], pred_ms) ** 0.5,
                R2=r2_score(objetivo(df), pred_log), top1=e["top1"], sobrecosto=e["sobrecosto"])

desarrollo = pd.concat([entrena, valida])   # todo lo que no es prueba
folds = []
pred_oof = np.zeros(len(desarrollo))
for k, (i_tr, i_te) in enumerate(GroupKFold(n_splits=5).split(desarrollo, groups=desarrollo["consulta_id"])):
    tr, te = desarrollo.iloc[i_tr], desarrollo.iloc[i_te]
    m = lgb.LGBMRegressor(**PARAMETROS_BASE).fit(tr[CARACTERISTICAS], objetivo(tr))
    pred_oof[i_te] = m.predict(te[CARACTERISTICAS])
    folds.append(metricas(te, pred_oof[i_te]))
folds = pd.DataFrame(folds)
folds.index = [f"fold {i+1}" for i in range(len(folds))]
print(folds.round(3).to_string())
print(f"\nPromedio ± desviación:  MAE = {folds.MAE.mean():.1f} ± {folds.MAE.std():.1f} ms | "
      f"R² = {folds.R2.mean():.3f} ± {folds.R2.std():.3f} | top-1 = {folds.top1.mean():.0%} ± {folds.top1.std():.0%}")

In [ ]:
# BLOQUE 7.3 — Busca la mejor configuración del modelo
GRILLA = [dict(num_leaves=nl, learning_rate=lr, min_child_samples=mc)
          for nl in (15, 31, 63) for lr in (0.03, 0.1) for mc in (10, 30)]

pruebas_hp = []
for p in GRILLA:
    m = lgb.LGBMRegressor(**{**PARAMETROS_BASE, **p, "n_estimators": 3000})
    m.fit(entrena[CARACTERISTICAS], objetivo(entrena),
          eval_set=[(valida[CARACTERISTICAS], objetivo(valida))], eval_metric="l1",
          callbacks=[lgb.early_stopping(50, verbose=False)])
    pred = m.predict(valida[CARACTERISTICAS])
    pruebas_hp.append({**p, "árboles": m.best_iteration_,
                       "error validación (log)": mean_absolute_error(objetivo(valida), pred),
                       "top-1 validación": evaluar_eleccion(valida, pred)["top1"]})

pruebas_hp = pd.DataFrame(pruebas_hp).sort_values("error validación (log)").reset_index(drop=True)
print(pruebas_hp.round(4).to_string())
MEJORES = pruebas_hp.iloc[0][["num_leaves", "learning_rate", "min_child_samples", "árboles"]].to_dict()
print("\nMejor combinación:", {k: (float(v) if k == "learning_rate" else int(v)) for k, v in MEJORES.items()})

In [ ]:
# BLOQUE 7.4 — Entrena el modelo final y lo prueba con consultas que no vio
PARAMETROS_FINALES = {**PARAMETROS_BASE,
                      "num_leaves": int(MEJORES["num_leaves"]),
                      "learning_rate": float(MEJORES["learning_rate"]),
                      "min_child_samples": int(MEJORES["min_child_samples"]),
                      "n_estimators": max(50, int(MEJORES["árboles"] * 1.2))}
modelo = lgb.LGBMRegressor(**PARAMETROS_FINALES).fit(desarrollo[CARACTERISTICAS], objetivo(desarrollo))

pred_prueba_log = modelo.predict(prueba[CARACTERISTICAS])
pred_prueba_ms = np.expm1(pred_prueba_log)
FINAL = metricas(prueba, pred_prueba_log)
print("== Modelo final sobre la prueba (consultas que nunca vio) ==")
print(f"MAE  = {FINAL['MAE']:.2f} ms   (error promedio)")
print(f"RMSE = {FINAL['RMSE']:.2f} ms   (castiga más los errores grandes)")
print(f"R²   = {FINAL['R2']:.3f}       (en escala log; 1 = perfecto)")
print(f"Comparado con la validación cruzada: R² {folds.R2.mean():.3f} ± {folds.R2.std():.3f}")

lo = max(0.01, min(prueba["tiempo_ms"].min(), pred_prueba_ms.min()))
hi = max(prueba["tiempo_ms"].max(), pred_prueba_ms.max())
plt.figure(figsize=(5.5, 5.5))
plt.scatter(prueba["tiempo_ms"], pred_prueba_ms, s=16, alpha=0.6, color="#2563eb")
plt.plot([lo, hi], [lo, hi], "r--", label="predicción perfecta")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Tiempo real (ms)"); plt.ylabel("Tiempo predicho (ms)")
plt.title("Modelo final: predicción vs realidad")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# BLOQUE 7.5 — En qué se fija el modelo
importancia = pd.Series(modelo.booster_.feature_importance(importance_type="gain"),
                        index=CARACTERISTICAS).sort_values(ascending=False)
importancia = importancia / importancia.sum()
print((importancia.head(10) * 100).round(1).astype(str).add(" %").to_string())

importancia.head(12)[::-1].plot.barh(figsize=(7, 5), color="#ea580c")
plt.xlabel("parte de la ganancia total"); plt.title("Qué mira el modelo para decidir")
plt.tight_layout(); plt.show()

In [ ]:
# BLOQUE 7.6 — Prueba con tiempos barajados: si el modelo aprende de verdad, aquí falla
rng = np.random.default_rng(SEMILLA)
barajadas = []
for r in range(5):
    y_falso = rng.permutation(objetivo(desarrollo).values)
    m = lgb.LGBMRegressor(**PARAMETROS_FINALES).fit(desarrollo[CARACTERISTICAS], y_falso)
    barajadas.append(metricas(prueba, m.predict(prueba[CARACTERISTICAS])))
barajadas = pd.DataFrame(barajadas)
AZAR_TOP1 = evaluar_eleccion(prueba, np.random.default_rng(SEMILLA).random(len(prueba)))["top1"]

print("                         R²      top-1")
print(f"Modelo real:           {FINAL['R2']:6.3f}   {FINAL['top1']:5.0%}")
print(f"Con tiempos barajados: {barajadas.R2.mean():6.3f}   {barajadas.top1.mean():5.0%}   (promedio de 5 intentos)")
print(f"Elegir al azar:            -     {AZAR_TOP1:5.0%}")

In [ ]:
# BLOQUE 7.7 — Prueba con una base que el modelo nunca vio
filas_lodo = []
for b in sorted(datos.base.unique()):
    otras, nueva = datos[datos.base != b], datos[datos.base == b]
    m = lgb.LGBMRegressor(**PARAMETROS_FINALES).fit(otras[CARACTERISTICAS], objetivo(otras))
    p = m.predict(nueva[CARACTERISTICAS])
    e_modelo, e_formula = evaluar_eleccion(nueva, p), evaluar_eleccion(nueva, nueva["raiz_costo"])
    nativo = sum(tiempos_nativos[q] for q in nueva.consulta_id.unique())
    filas_lodo.append({"base nueva": b, "consultas": nueva.consulta_id.nunique(),
                       "R²": r2_score(objetivo(nueva), p),
                       "top-1 modelo": e_modelo["top1"], "top-1 fórmula": e_formula["top1"],
                       "mejora modelo vs PG": 1 - e_modelo["tiempo_elegido"] / nativo,
                       "mejora fórmula vs PG": 1 - e_formula["tiempo_elegido"] / nativo})
LODO = pd.DataFrame(filas_lodo)
display(LODO.style.format({"R²": "{:.3f}", "top-1 modelo": "{:.0%}", "top-1 fórmula": "{:.0%}",
                           "mejora modelo vs PG": "{:+.1%}", "mejora fórmula vs PG": "{:+.1%}"}).hide(axis="index"))
print(f"Promedio en bases nuevas: top-1 modelo {LODO['top-1 modelo'].mean():.0%} vs fórmula {LODO['top-1 fórmula'].mean():.0%}")
fin_bloque("Bloque 7 - entrenamiento", inicio_bloque7)

In [ ]:
# BLOQUE 8 — V1: el modelo elige el plan de cada consulta
NATIVO = "(PostgreSQL decide)"

def puntaje_prudente(df, pred_log, umbral):
    p = np.expm1(np.asarray(pred_log, dtype=float)).copy()
    p[(df["orden"] == NATIVO).values] *= (1 - umbral)
    return p

m_val = lgb.LGBMRegressor(**{**PARAMETROS_FINALES, "n_estimators": int(MEJORES["árboles"])})
m_val.fit(entrena[CARACTERISTICAS], objetivo(entrena))
pred_val = m_val.predict(valida[CARACTERISTICAS])
umbrales = pd.Series({u: evaluar_eleccion(valida, puntaje_prudente(valida, pred_val, u))["tiempo_elegido"]
                      for u in [0, 0.05, 0.1, 0.2, 0.3, 0.5]})
UMBRAL = float(max(u for u, t in umbrales.items() if t <= umbrales.min() * 1.01))
print("Tiempo total en validación según el umbral (ms):", umbrales.round(0).to_dict())
print(f"Umbral elegido: {UMBRAL:.0%} (el modelo cambia el plan solo si predice al menos {UMBRAL:.0%} de mejora)\n")

DECISION = evaluar_eleccion(prueba, puntaje_prudente(prueba, pred_prueba_log, UMBRAL))
nativo_prueba = sum(tiempos_nativos[q] for q in prueba.consulta_id.unique())
print(f"Consultas de prueba: {prueba.consulta_id.nunique()}")
print(f"top-1: {DECISION['top1']:.0%} | sobrecosto mediano: {DECISION['sobrecosto']:+.1%}")
print(f"Tiempo total del workload de prueba:")
print(f"  PostgreSQL solo (nativo):    {nativo_prueba:9,.0f} ms")
print(f"  Plan elegido por el modelo:  {DECISION['tiempo_elegido']:9,.0f} ms")
print(f"  Mejor plan posible:          {DECISION['tiempo_mejor']:9,.0f} ms")

ejemplo_id = por_consulta.loc[list(ids_prueba)]["peor/mejor"].idxmax()
ejemplo = prueba[prueba.consulta_id == ejemplo_id].assign(
    predicho_ms=pred_prueba_ms[prueba.consulta_id.values == ejemplo_id])
ejemplo = ejemplo[["orden", "raiz_costo", "predicho_ms", "tiempo_ms"]].sort_values("predicho_ms")
usar_base(BASE_DE[ejemplo_id])
print(f"\nEjemplo, consulta {ejemplo_id} (base {BASE_DE[ejemplo_id]}): {construir_sql(*CONSULTAS[ejemplo_id])}")
print(f"PostgreSQL solo tardó {tiempos_nativos[ejemplo_id]:.2f} ms. El modelo elige la primera fila:")
print(ejemplo.round(1).to_string(index=False))

In [ ]:
# BLOQUE 9 — V1: el modelo contra la fórmula de PostgreSQL
azar = np.random.default_rng(SEMILLA).random(len(prueba))
COMPARACION = pd.DataFrame({
    "al azar":                evaluar_eleccion(prueba, azar),
    "fórmula de costos (PG)": evaluar_eleccion(prueba, prueba["raiz_costo"]),
    "modelo ML":              evaluar_eleccion(prueba, puntaje_prudente(prueba, pred_prueba_log, UMBRAL)),
}).T
COMPARACION["vs mejor posible"] = COMPARACION["tiempo_elegido"] / COMPARACION["tiempo_mejor"] - 1

def spearman(puntaje):
    df = prueba.assign(p=np.asarray(puntaje))
    return df.groupby("consulta_id").apply(lambda g: g["p"].corr(g["tiempo_ms"], method="spearman")).mean()
COMPARACION["ordena bien (Spearman)"] = [spearman(azar), spearman(prueba["raiz_costo"]), spearman(pred_prueba_log)]

display(COMPARACION[["top1", "sobrecosto", "tiempo_elegido", "vs mejor posible", "ordena bien (Spearman)"]]
      .rename(columns={"top1": "top-1", "sobrecosto": "sobrecosto mediano", "tiempo_elegido": "tiempo total (ms)"})
      .style.format({"top-1": "{:.0%}", "sobrecosto mediano": "{:+.1%}", "tiempo total (ms)": "{:,.0f}",
                     "vs mejor posible": "{:+.1%}", "ordena bien (Spearman)": "{:.2f}"}))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
colores = ["#9ca3af", "#16a34a", "#2563eb"]
ax[0].bar(COMPARACION.index, COMPARACION["top1"] * 100, color=colores)
ax[0].set_ylabel("% de consultas"); ax[0].set_title("Elige el mejor plan (top-1)")
ax[1].bar(COMPARACION.index, COMPARACION["tiempo_elegido"], color=colores)
ax[1].axhline(DECISION["tiempo_mejor"], color="k", ls="--", label="mejor posible")
ax[1].set_ylabel("ms"); ax[1].set_title("Tiempo total del workload de prueba"); ax[1].legend()
for a in ax: a.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

from scipy.stats import wilcoxon

todas = pd.concat([desarrollo, prueba])
pred_todas = np.concatenate([pred_oof, pred_prueba_log])

def elegido_por_consulta(df, puntaje):
    df = df.assign(p=np.asarray(puntaje))
    return df.loc[df.groupby("consulta_id")["p"].idxmin()].set_index("consulta_id")["tiempo_ms"]

t_modelo  = elegido_por_consulta(todas, puntaje_prudente(todas, pred_todas, UMBRAL))
t_formula = elegido_por_consulta(todas, todas["raiz_costo"]).loc[t_modelo.index]
t_nativo  = pd.Series(tiempos_nativos).loc[t_modelo.index]

for nombre, otro in [("fórmula de costos", t_formula), ("PostgreSQL solo", t_nativo)]:
    p = wilcoxon(otro, t_modelo, alternative="greater").pvalue
    gana, empata = (t_modelo < otro * 0.95).sum(), ((t_modelo - otro).abs() <= otro * 0.05).sum()
    print(f"Modelo vs {nombre:17s}: el modelo es más rápido en {gana}, empata en {empata}, "
          f"es más lento en {len(t_modelo) - gana - empata} de {len(t_modelo)} consultas | "
          f"p-valor = {p:.4f} -> {'SIGNIFICATIVO' if p < 0.05 else 'no significativo'}")

t_mejor = todas.groupby("consulta_id")["tiempo_ms"].min().loc[t_modelo.index]
TOTALES = dict(nativo=t_nativo.sum(), formula=t_formula.sum(), modelo=t_modelo.sum(), mejor=t_mejor.sum())
print(f"\nTiempo total de las {len(t_modelo)} consultas: PostgreSQL solo {TOTALES['nativo']:,.0f} ms | "
      f"fórmula {TOTALES['formula']:,.0f} ms | modelo {TOTALES['modelo']:,.0f} ms "
      f"({1 - TOTALES['modelo'] / TOTALES['nativo']:+.1%}) | mejor posible {TOTALES['mejor']:,.0f} ms")

por_base_v1 = pd.DataFrame({"base": [BASE_DE[q] for q in t_modelo.index], "nativo": t_nativo.values,
                            "formula": t_formula.values, "modelo": t_modelo.values})
por_base_v1 = por_base_v1.groupby("base").sum()
por_base_v1["mejora modelo"] = 1 - por_base_v1.modelo / por_base_v1.nativo
por_base_v1["mejora fórmula"] = 1 - por_base_v1.formula / por_base_v1.nativo
display(por_base_v1.style.format({"nativo": "{:,.0f}", "formula": "{:,.0f}", "modelo": "{:,.0f}",
                                  "mejora modelo": "{:+.1%}", "mejora fórmula": "{:+.1%}"}))

In [ ]:
# BLOQUE 10.1 — V2: crea cada índice, mide cuánto ayuda y arma los datos
def quitar_indices_secundarios(cur):
    cur.execute("SELECT indexname FROM pg_indexes WHERE schemaname='public' AND indexname NOT LIKE '%_pkey'")
    for (nombre,) in cur.fetchall():
        cur.execute(f'DROP INDEX IF EXISTS "{nombre}"')

def columnas_de(tablas, filtros):
    usadas = {}
    for f in filtros:
        for t, c in re.findall(r"\b([a-z_]\w*)\.([a-z_]\w*)\b", f):
            usadas[(t, c)] = "filtro"
    for arista, condicion in ARISTAS.items():
        if arista <= set(tablas):
            for t, c in re.findall(r"\b([a-z_]\w*)\.([a-z_]\w*)\b", condicion):
                usadas.setdefault((t, c), "join")
    return {k: v for k, v in usadas.items() if k[1] not in CLAVES_PRIMARIAS}

def estadisticas(cur):
    cur.execute("SELECT relname, reltuples, relpages FROM pg_class WHERE relname = ANY(%s) AND relkind = 'r'", (TABLAS,))
    tablas = {t: dict(filas=max(f, 1), paginas=max(p, 1)) for t, f, p in cur.fetchall()}
    cur.execute("SELECT tablename, attname, n_distinct, correlation, avg_width FROM pg_stats WHERE schemaname='public'")
    columnas = {}
    for t, c, nd, corr, ancho in cur.fetchall():
        filas = tablas.get(t, {"filas": 1})["filas"]
        distintos = -nd * filas if nd < 0 else nd
        columnas[(t, c)] = dict(distintos=min(1.0, distintos / filas), correlacion=abs(corr or 0), ancho=ancho)
    return tablas, columnas

def lecturas_por_tabla(plan):
    salida = {}
    def recorrer(nodo):
        if "Relation Name" in nodo and nodo["Relation Name"] not in salida:
            salida[nodo["Relation Name"]] = nodo
        for hijo in nodo.get("Plans", []):
            recorrer(hijo)
    recorrer(plan)
    return salida

def caracteristicas_indice(plan, tablas_q, tabla, columna, uso, T, C):
    nodo = lecturas_por_tabla(plan).get(tabla, {})
    filas = T[tabla]["filas"]
    return dict(
        log_filas_tabla=np.log10(filas),                                         # tabla chica o grande
        es_filtro=int(uso == "filtro"), es_join=int(uso == "join"),              # para qué se usa la columna
        selectividad=min(1.0, (nodo.get("Plan Rows") or filas) / filas),         # qué fracción de filas sobrevive
        lee_entera=int(nodo.get("Node Type") == "Seq Scan"),                    # hoy la recorre completa
        parte_del_costo=(nodo.get("Total Cost") or 0) / max(plan["Total Cost"], 1e-3),  # cuánto pesa leer esa tabla
        distintos=C.get((tabla, columna), {}).get("distintos", 0),             # 1 = todos distintos
        correlacion=C.get((tabla, columna), {}).get("correlacion", 0),         # 1 = datos ordenados en disco
        n_tablas=len(tablas_q),
        es_tabla_mas_grande=int(filas >= max(T[x]["filas"] for x in tablas_q)),
    )

inicio = time.time()
consultas_validas = sorted(datos["consulta_id"].unique())
pares, base_sin, planes_con_indice = [], {}, []
STATS = {}   # base -> (estadísticas de tablas, estadísticas de columnas)
grupos = list(por_base(consultas_validas))
for hechas, (base, lista) in enumerate(grupos, 1):
    usar_base(base)
    with conexion.cursor() as cur:
        configurar_sesion(cur, forzar=False)
        quitar_indices_secundarios(cur); cur.execute("ANALYZE")
        STATS[base] = T_STATS, C_STATS = estadisticas(cur)

        usos = {}   # (tabla, columna) -> consultas que la usan
        for q in lista:
            tablas, filtros = CONSULTAS[q]
            sql = construir_sql(tablas, filtros)
            try:
                base_sin[q] = dict(plan=plan_estimado(cur, sql), t=tiempo_real(cur, sql))
            except Exception:
                continue
            for col, uso in columnas_de(tablas, filtros).items():
                usos.setdefault(col, []).append((q, uso))

        for (t, c), lista_q in sorted(usos.items()):
            cur.execute(f"CREATE INDEX idx_{t}_{c} ON {t}({c})")
            cur.execute("SELECT pg_relation_size(%s)", (f"idx_{t}_{c}",))
            mb_real = cur.fetchone()[0] / 2**20
            for q, uso in lista_q:
                tablas, filtros = CONSULTAS[q]
                sql_q = construir_sql(tablas, filtros)
                try:
                    plan_con = plan_estimado(cur, sql_q)
                    t_con = tiempo_real(cur, sql_q)
                    planes_con_indice.append({**extraer_caracteristicas(plan_con, len(tablas)),
                                              "consulta_id": q, "base": base, "indice": f"{t}({c})", "tiempo_ms": t_con})
                except Exception:
                    t_con = base_sin[q]["t"]
                fila = caracteristicas_indice(base_sin[q]["plan"], tablas, t, c, uso, T_STATS, C_STATS)
                fila.update(consulta_id=q, base=base, indice=f"{t}({c})", mb_real=mb_real,
                            t_sin=base_sin[q]["t"], t_con=t_con,
                            mejora=float(np.clip(1 - t_con / max(base_sin[q]["t"], 1e-3), -1, 1)))
                pares.append(fila)
            cur.execute(f"DROP INDEX idx_{t}_{c}")
    avance(hechas, len(grupos), inicio, f"base {base:15s} lista")

pares = pd.DataFrame(pares)
CARACT_INDICE = ["log_filas_tabla", "es_filtro", "es_join", "selectividad", "lee_entera",
                 "parte_del_costo", "distintos", "correlacion", "n_tablas", "es_tabla_mas_grande"]
print(f"Dataset de índices: {len(pares)} pares (consulta, índice) | "
      f"{pares[['base', 'indice']].drop_duplicates().shape[0]} columnas candidatas en {pares.base.nunique()} bases")
print(f"El índice ayudó más de 10% en el {(pares.mejora > .1).mean():.0%} de los pares")
print(pares.groupby(["base", "indice"]).agg(consultas=("mejora", "size"), mejora_media=("mejora", "mean"),
                                            mb=("mb_real", "first"))
      .sort_values("mejora_media", ascending=False).head(15).round(3).to_string())
fin_bloque("Bloque 10.1 - dataset de índices", inicio)

In [ ]:
# BLOQUE 10.2 — V2: entrena el modelo de índices
PARAM_INDICE = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
                    subsample=0.9, subsample_freq=1, colsample_bytree=0.9, random_state=SEMILLA, verbosity=-1)

pares_des = pares[~pares.consulta_id.isin(ids_prueba)].reset_index(drop=True)
pares_pru = pares[pares.consulta_id.isin(ids_prueba)].reset_index(drop=True)

def eval_indice(df, pred):
    return dict(MAE=mean_absolute_error(df.mejora, pred),
                acierto=((pred > .1) == (df.mejora > .1)).mean())

folds_idx = []
for i_tr, i_te in GroupKFold(n_splits=5).split(pares_des, groups=pares_des.consulta_id):
    m = lgb.LGBMRegressor(**PARAM_INDICE).fit(pares_des.iloc[i_tr][CARACT_INDICE], pares_des.iloc[i_tr].mejora)
    folds_idx.append(eval_indice(pares_des.iloc[i_te], m.predict(pares_des.iloc[i_te][CARACT_INDICE])))
folds_idx = pd.DataFrame(folds_idx)

modelo_indices = lgb.LGBMRegressor(**PARAM_INDICE).fit(pares_des[CARACT_INDICE], pares_des.mejora)
pred_pares = modelo_indices.predict(pares_pru[CARACT_INDICE])
FINAL_IDX = eval_indice(pares_pru, pred_pares)

falso = lgb.LGBMRegressor(**PARAM_INDICE).fit(pares_des[CARACT_INDICE],
                                              np.random.default_rng(SEMILLA).permutation(pares_des.mejora.values))
BARAJADO_IDX = eval_indice(pares_pru, falso.predict(pares_pru[CARACT_INDICE]))
SIEMPRE_NO = ((pares_pru.mejora > .1) == False).mean()

print(f"Validación cruzada:   MAE = {folds_idx.MAE.mean():.3f} ± {folds_idx.MAE.std():.3f} | "
      f"acierto = {folds_idx.acierto.mean():.0%} ± {folds_idx.acierto.std():.0%}")
print(f"Prueba (no vista):    MAE = {FINAL_IDX['MAE']:.3f} | acierto = {FINAL_IDX['acierto']:.0%}")
print(f"Tiempos barajados:    MAE = {BARAJADO_IDX['MAE']:.3f} | acierto = {BARAJADO_IDX['acierto']:.0%}")
print(f"Decir siempre 'no ayuda': acierto = {SIEMPRE_NO:.0%}")

imp = pd.Series(modelo_indices.booster_.feature_importance(importance_type="gain"), index=CARACT_INDICE)
(imp / imp.sum()).sort_values().plot.barh(figsize=(6, 4), color="#ea580c")
plt.title("Qué mira el modelo de índices"); plt.xlabel("parte de la ganancia total")
plt.tight_layout(); plt.show()

In [ ]:
# BLOQUE 10.3 — V2: el modelo recomienda índices y se comprueba
def mb_estimado(base, t, c):
    T, C = STATS[base]
    return T[t]["filas"] * (C[(t, c)]["ancho"] + 16) * 1.15 / 2**20

def medir_con(cur, lista_ddl, consultas_q, planes=None):
    quitar_indices_secundarios(cur)
    for ddl in lista_ddl:
        cur.execute(ddl)
    cur.execute("ANALYZE")
    cur.execute("SELECT coalesce(sum(pg_relation_size(indexrelid)), 0) FROM pg_index i "
                "JOIN pg_class c ON c.oid = i.indexrelid WHERE NOT i.indisprimary AND c.relnamespace = 'public'::regnamespace")
    mb = float(cur.fetchone()[0]) / 2**20
    total = 0.0
    for q in consultas_q:
        candidato = (planes or {}).get(q, (None, ()))
        aplicar_candidato(cur, candidato)
        try:
            total += tiempo_real(cur, sql_de(q, candidato))
        except Exception:
            total += LIMITE_MS
    limpiar_sesion(cur)
    return total, mb

def aplicar_candidato(cur, candidato):
    orden, apagados = candidato
    configurar_sesion(cur, forzar=orden is not None)
    cur.execute("RESET enable_hashjoin; RESET enable_mergejoin; RESET enable_nestloop;")
    for metodo in apagados:
        cur.execute(f"SET enable_{metodo}=off;")

def limpiar_sesion(cur):
    cur.execute("RESET enable_hashjoin; RESET enable_mergejoin; RESET enable_nestloop;")
    configurar_sesion(cur, forzar=False)

def sql_de(q, candidato):
    tablas, filtros = CONSULTAS[q]
    return construir_sql(candidato[0] or tablas, filtros)

def mejoras_predichas(base, consultas_q, m_idx):
    salida = {}
    for q in consultas_q:
        tablas, filtros = CONSULTAS[q]
        for (t, c), uso in columnas_de(tablas, filtros).items():
            x = caracteristicas_indice(base_sin[q]["plan"], tablas, t, c, uso, *STATS[base])
            salida[(q, (t, c))] = float(m_idx.predict(pd.DataFrame([x])[CARACT_INDICE])[0])
    return salida

def ganancias_predichas(base, consultas_q, m_plan, m_idx):
    tiempo = {q: float(np.expm1(m_plan.predict(pd.DataFrame(
        [extraer_caracteristicas(base_sin[q]["plan"], len(CONSULTAS[q][0]))])[CARACTERISTICAS]))[0])
        for q in consultas_q}
    ganancia = {}
    for (q, k), mejora in mejoras_predichas(base, consultas_q, m_idx).items():
        ganancia[k] = ganancia.get(k, 0.0) + max(mejora, 0) * tiempo[q]
    return ganancia

def elegir_indices(base, ganancia, presupuesto):
    limite = presupuesto * sum(mb_estimado(base, *k) for k in ganancia)
    elegidos, usado = [], 0.0
    for k in sorted(ganancia, key=lambda k: ganancia[k] / mb_estimado(base, *k), reverse=True):
        if ganancia[k] <= 0 or usado + mb_estimado(base, *k) > limite:
            continue
        elegidos.append(k); usado += mb_estimado(base, *k)
    return elegidos, limite

MEJORA_REAL = {(r.consulta_id, tuple(r.indice[:-1].split("("))): r.mejora for r in pares.itertuples()}

def ganancia_real(consultas_q, indices):
    return sum(base_sin[q]["t"] * max([0.0] + [MEJORA_REAL.get((q, k), 0.0) for k in indices]) for q in consultas_q)

def puntaje_equilibrio(filas):
    g = sum(f["ganado"] for f in filas) / max(sum(f["ganado_todos"] for f in filas), 1e-9)
    e = sum(f["mb"] for f in filas) / max(sum(f["mb_todos"] for f in filas), 1e-9)
    return g - e, g, e

inicio = time.time()
pares_ent = pares[pares.consulta_id.isin(ids_entrena)]
m_idx_val = lgb.LGBMRegressor(**PARAM_INDICE).fit(pares_ent[CARACT_INDICE], pares_ent.mejora)
grupos_valida = [(b, [q for q in qs if q in base_sin]) for b, qs in por_base(sorted(ids_valida))]
GANANCIA_VAL = {}
for base, consultas_val in grupos_valida:
    usar_base(base)
    GANANCIA_VAL[base] = ganancias_predichas(base, consultas_val, m_val, m_idx_val)
prueba_presupuesto = []
for p in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]:
    filas = []
    for base, consultas_val in grupos_valida:
        usar_base(base)
        todos = list(GANANCIA_VAL[base])
        elegidos, _ = elegir_indices(base, GANANCIA_VAL[base], p)
        filas.append(dict(ganado=ganancia_real(consultas_val, elegidos), ganado_todos=ganancia_real(consultas_val, todos),
                          mb=sum(mb_estimado(base, *k) for k in elegidos), mb_todos=sum(mb_estimado(base, *k) for k in todos)))
    puntaje, g, e = puntaje_equilibrio(filas)
    prueba_presupuesto.append(dict(presupuesto=p, mejora_conseguida=g, espacio_usado=e, puntaje=puntaje))
prueba_presupuesto = pd.DataFrame(prueba_presupuesto)
PRESUPUESTO = float(prueba_presupuesto.loc[prueba_presupuesto.puntaje.round(3).idxmax(), "presupuesto"])
print("Presupuesto aprendido en validación (mejora y espacio como % del máximo posible):")
display(prueba_presupuesto.style.format({"presupuesto": "{:.0%}", "mejora_conseguida": "{:.0%}",
                                         "espacio_usado": "{:.0%}", "puntaje": "{:+.3f}"}).hide(axis="index"))
print(f"Presupuesto elegido: {PRESUPUESTO:.0%} del espacio de todos los candidatos\n")

RECOMENDADOS, CANDIDATOS_BASE, filas_v2 = {}, {}, []
grupos_prueba = [(b, [q for q in qs if q in base_sin]) for b, qs in por_base(sorted(ids_prueba))]
for hechas, (base, consultas_prueba) in enumerate(grupos_prueba, 1):
    usar_base(base)
    candidatos = ganancias_predichas(base, consultas_prueba, modelo, modelo_indices)
    elegidos, limite = elegir_indices(base, candidatos, PRESUPUESTO)
    RECOMENDADOS[base], CANDIDATOS_BASE[base] = elegidos, candidatos
    elegidos_falso, _ = elegir_indices(base, ganancias_predichas(base, consultas_prueba, modelo, falso), PRESUPUESTO)
    print(f"\n[{base}] {len(consultas_prueba)} consultas de prueba | {len(candidatos)} candidatos | "
          f"presupuesto {limite:.1f} MB | recomendados por el modelo:")
    for t, c in elegidos:
        print(f"   CREATE INDEX ON {t}({c});   -- ~{mb_estimado(base, t, c):.1f} MB, ganancia predicha {candidatos[(t, c)]:.0f} ms")

    opciones = {
        "sin índices secundarios":    [],
        "índices típicos (bloque 3)": INDICES_BASE,
        "recomendados por el modelo": [f"CREATE INDEX ON {t}({c})" for t, c in elegidos],
        "control: modelo barajado":   [f"CREATE INDEX ON {t}({c})" for t, c in elegidos_falso],
        "todos los candidatos":       [f"CREATE INDEX ON {t}({c})" for t, c in candidatos],
    }
    with conexion.cursor() as cur:
        for nombre, ddl in opciones.items():
            total, mb = medir_con(cur, ddl, consultas_prueba)
            filas_v2.append(dict(base=base, opción=nombre, índices=len(ddl), espacio_mb=mb, tiempo_ms=total))
        medir_con(cur, INDICES_BASE, [])      # dejar la base como en el bloque 3
    avance(hechas, len(grupos_prueba), inicio, f"base {base:15s} verificada")

DETALLE_V2 = pd.DataFrame(filas_v2)
VERSION2 = DETALLE_V2.groupby("opción", sort=False)[["índices", "espacio_mb", "tiempo_ms"]].sum().reset_index()
VERSION2["mejora vs sin índices"] = 1 - VERSION2.tiempo_ms / VERSION2.tiempo_ms.iloc[0]
VERSION2["mejora por MB"] = VERSION2["mejora vs sin índices"] / VERSION2.espacio_mb.replace(0, np.nan)
print(f"\nVerificación real, suma de todas las bases:")
display(VERSION2.style.format({"espacio_mb": "{:.1f}", "tiempo_ms": "{:,.0f}",
                               "mejora vs sin índices": "{:+.1%}", "mejora por MB": "{:+.1%}"}).hide(axis="index"))
fin_bloque("Bloque 10.3 - recomendación de índices", inicio)

In [ ]:
# BLOQUE 11.1 — V3: arma una base gemela con las estadísticas del servidor
# (HypoPG se instala solo en la gemela, nunca en el servidor)
def leer_servidor(c_lectura):
    with c_lectura.cursor() as cur:
        cur.execute("SELECT table_name, column_name, data_type FROM information_schema.columns "
                    "WHERE table_schema = 'public' ORDER BY table_name, ordinal_position")
        columnas = {}
        for t, c, tipo in cur.fetchall():
            columnas.setdefault(t, []).append((c, tipo))
        cur.execute("SELECT tc.table_name, kcu.column_name FROM information_schema.table_constraints tc "
                    "JOIN information_schema.key_column_usage kcu ON kcu.constraint_name = tc.constraint_name "
                    "AND kcu.table_schema = tc.table_schema WHERE tc.table_schema = 'public' AND tc.constraint_type = 'PRIMARY KEY'")
        claves = dict(cur.fetchall())
        cur.execute("SELECT indexdef FROM pg_indexes WHERE schemaname = 'public' AND indexname NOT LIKE '%%_pkey'")
        indices = [r[0] for r in cur.fetchall()]
        cur.execute("SELECT relname, reltuples FROM pg_class WHERE relnamespace = 'public'::regnamespace AND relkind = 'r'")
        filas = {t: int(max(n, 0)) for t, n in cur.fetchall()}
        cur.execute("SELECT tablename, attname, null_frac, n_distinct, most_common_vals::text::text[], most_common_freqs, "
                    "histogram_bounds::text::text[], correlation, avg_width FROM pg_stats WHERE schemaname = 'public'")
        stats = {(r[0], r[1]): dict(nulos=r[2], distintos=r[3], frecuentes=r[4] or [], frecuencias=r[5] or [],
                                    histograma=r[6] or [], correlacion=r[7] or 0, ancho=r[8]) for r in cur.fetchall()}
        cur.execute("SELECT extname FROM pg_extension")
        extensiones = [r[0] for r in cur.fetchall()]
    return dict(columnas=columnas, claves=claves, indices=indices, filas=filas, stats=stats, extensiones=extensiones)

def a_numero(valores, tipo):
    if tipo == "date":
        return np.array(valores, dtype="datetime64[D]").astype(np.int64).astype(float)
    return np.array(valores, dtype=float)

def de_numero(v, tipo):
    if tipo == "date":
        return np.round(v).astype(np.int64).astype("datetime64[D]").astype(str)
    if tipo in ("integer", "bigint", "smallint"):
        return np.round(v).astype(np.int64)
    return np.round(v, 2)

def columna_gemela(st, tipo, n, es_clave, rng):
    texto = tipo in ("text", "character varying")
    hist = st["histograma"]
    if es_clave:
        lo, hi = a_numero([hist[0], hist[-1]], tipo)
        return de_numero(lo + np.floor(np.arange(n) * max(1.0, (hi - lo) / max(n - 1, 1))), tipo)
    if texto and st["distintos"] < -0.5:
        return np.array([f"g{i}".ljust(max(1, st["ancho"] - 1), "x") for i in range(n)], dtype=object)
    partes = []
    for valor, frecuencia in zip(st["frecuentes"], st["frecuencias"]):
        partes.append(np.full(int(round(frecuencia * n)), valor, dtype=object))
    resto = max(0, n - sum(len(p) for p in partes) - int(round(st["nulos"] * n)))
    if resto and len(hist) >= 2:
        tramo = rng.integers(0, len(hist) - 1, resto)
        if texto:
            partes.append(np.array(hist, dtype=object)[tramo + rng.integers(0, 2, resto)])
        else:
            limites = a_numero(hist, tipo)
            v = limites[tramo] + rng.random(resto) * (limites[tramo + 1] - limites[tramo])
            partes.append(de_numero(v, tipo).astype(object))
    elif resto and st["frecuentes"]:
        p = np.array(st["frecuencias"], dtype=float)
        partes.append(np.array(st["frecuentes"], dtype=object)[rng.choice(len(p), resto, p=p / p.sum())])
    valores = np.concatenate(partes) if partes else np.array([], dtype=object)
    valores = np.concatenate([valores, np.full(max(0, n - len(valores)), None, dtype=object)])[:n]
    rng.shuffle(valores)
    corr = st["correlacion"]
    if abs(corr) >= 0.3 and not texto:
        no_nulos = np.array([v is not None for v in valores])
        orden = np.sort(a_numero(valores[no_nulos].tolist(), tipo) if tipo == "date" else valores[no_nulos].astype(float))
        if corr < 0:
            orden = orden[::-1]
        desordenar = rng.choice(len(orden), int(len(orden) * (1 - abs(corr))), replace=False)
        orden[desordenar] = orden[rng.permutation(desordenar)]
        valores = np.concatenate([de_numero(orden, tipo).astype(object), valores[~no_nulos]])
    return valores

def armar_gemela(base):
    c_lectura = conectar(f"optim_{base}", solo_lectura=True)
    info = leer_servidor(c_lectura)
    c_lectura.close()
    nombre_gemela = f"gemela_{base}"
    with conexion_admin.cursor() as cur:
        cur.execute(f"DROP DATABASE IF EXISTS {nombre_gemela}")
        cur.execute(f"CREATE DATABASE {nombre_gemela}")
    c = conectar(nombre_gemela)
    rng = np.random.default_rng(SEMILLA)
    with c.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS hypopg")
        for t, cols in info["columnas"].items():
            n = info["filas"].get(t, 0)
            ddl = [f"{col} {tipo}" + (" PRIMARY KEY" if info["claves"].get(t) == col else "") for col, tipo in cols]
            cur.execute(f"CREATE TABLE {t} ({', '.join(ddl)})")
            datos_t = {col: columna_gemela(info["stats"][(t, col)], tipo, n, info["claves"].get(t) == col, rng)
                       for col, tipo in cols if (t, col) in info["stats"]}
            copiar(cur, t, pd.DataFrame(datos_t).fillna("\\N"))
        for ddl in info["indices"]:
            cur.execute(ddl)
        cur.execute("ANALYZE")
    return c, info

def forma_del_plan(plan):
    salida = []
    def recorrer(nodo):
        salida.append((nodo.get("Node Type"), nodo.get("Relation Name")))
        for hijo in nodo.get("Plans", []):
            recorrer(hijo)
    recorrer(plan)
    return tuple(salida)

inicio = time.time()
GEMELAS, fidelidad = {}, []
for hechas, (base, consultas_prueba) in enumerate(grupos_prueba, 1):
    usar_base(base)
    with conexion.cursor() as cur:
        quitar_indices_secundarios(cur); cur.execute("ANALYZE")   # escenario de partida: sin índices secundarios
    GEMELAS[base], info = armar_gemela(base)
    iguales, razones = 0, []
    with conexion.cursor() as cur_s, GEMELAS[base].cursor() as cur_g:
        for q in consultas_prueba:
            sql = construir_sql(*CONSULTAS[q])
            p_s, p_g = plan_estimado(cur_s, sql), plan_estimado(cur_g, sql)
            iguales += forma_del_plan(p_s) == forma_del_plan(p_g)
            razones.append(p_g["Total Cost"] / max(p_s["Total Cost"], 1e-3))
    fidelidad.append(dict(base=base, consultas=len(consultas_prueba), mismo_plan=iguales / max(len(consultas_prueba), 1),
                          costo_gemela_vs_servidor=float(np.median(razones)),
                          hypopg_en_servidor="hypopg" in info["extensiones"]))
    avance(hechas, len(grupos_prueba), inicio, f"gemela {base:15s} lista")
FIDELIDAD = pd.DataFrame(fidelidad)
display(FIDELIDAD.style.format({"mismo_plan": "{:.0%}", "costo_gemela_vs_servidor": "{:.2f}"}).hide(axis="index"))
print("mismo_plan = % de consultas donde la gemela arma exactamente el mismo plan que el servidor")
print("costo_gemela_vs_servidor = 1.00 significa que la gemela estima el mismo costo")
print("hypopg_en_servidor = False: el servidor nunca tuvo HypoPG instalado")
fin_bloque("Bloque 11.1 - base gemela", inicio)

In [ ]:
# BLOQUE 11.2 — V3: el modelo elige índices y planes juntos en la gemela
inicio = time.time()
filas_extra = planes_con_indice + [{**extraer_caracteristicas(v["plan"], len(CONSULTAS[q][0])),
                                    "consulta_id": q, "base": BASE_DE[q], "indice": "-", "tiempo_ms": v["t"]}
                                   for q, v in base_sin.items()]
extra = pd.DataFrame(filas_extra).fillna(0)
extra_des, extra_pru = extra[~extra.consulta_id.isin(ids_prueba)], extra[extra.consulta_id.isin(ids_prueba)]
entrena_v3 = pd.concat([desarrollo, extra_des]).fillna(0)
modelo_v3 = lgb.LGBMRegressor(**PARAMETROS_FINALES).fit(entrena_v3[CARACTERISTICAS], objetivo(entrena_v3))
print(f"Modelo de la versión 3: {len(entrena_v3)} planes de entrenamiento "
      f"({len(desarrollo)} de la versión 1 + {len(extra_des)} con y sin índices)")
entrena_v3_val = pd.concat([entrena, extra[extra.consulta_id.isin(ids_entrena)]]).fillna(0)
m_v3_val = lgb.LGBMRegressor(**PARAMETROS_FINALES).fit(entrena_v3_val[CARACTERISTICAS], objetivo(entrena_v3_val))
v3_barajado = lgb.LGBMRegressor(**PARAMETROS_FINALES).fit(
    entrena_v3[CARACTERISTICAS], np.random.default_rng(SEMILLA).permutation(objetivo(entrena_v3).values))
print("Error al predecir planes de consultas de PRUEBA con y sin índice:")
for nombre, m in [("modelo versión 1", modelo), ("modelo versión 3", modelo_v3)]:
    p = m.predict(extra_pru[CARACTERISTICAS])
    print(f"   {nombre}: R² = {r2_score(objetivo(extra_pru), p):.3f} | "
          f"MAE = {mean_absolute_error(extra_pru.tiempo_ms, np.expm1(p)):.1f} ms")

def candidatos_de(q):
    tablas, _ = CONSULTAS[q]
    rng = random.Random(SEMILLA + int(q))
    return [(None, ())] + [(None, a) for a in APAGAR] + [(o, ()) for o in ordenes_conectados(tablas, MAX_ORDENES, rng)]

CACHE_V3, CONTADOR = {}, dict(explain_gemela=0, consultas_servidor=0)
def mejor_plan(cur_g, q, indices, m_plan):
    relevantes = tuple(sorted(set(indices) & set(columnas_de(*CONSULTAS[q]))))
    clave = (q, relevantes, id(m_plan))
    if clave in CACHE_V3:
        return CACHE_V3[clave]
    cur_g.execute("SELECT hypopg_reset()")
    for t, c in relevantes:
        cur_g.execute("SELECT * FROM hypopg_create_index(%s)", (f"CREATE INDEX ON {t}({c})",))
    filas, cands, vistos = [], [], set()
    for cand in candidatos_de(q):
        aplicar_candidato(cur_g, cand)
        try:
            plan = plan_estimado(cur_g, sql_de(q, cand)); CONTADOR["explain_gemela"] += 1
        except Exception:
            continue
        firma = json.dumps(plan, sort_keys=True)
        if firma in vistos:            # mismo plan que otro candidato: no hace falta repetirlo
            continue
        vistos.add(firma)
        filas.append(extraer_caracteristicas(plan, len(CONSULTAS[q][0]))); cands.append(cand)
    limpiar_sesion(cur_g)
    pred = np.expm1(m_plan.predict(pd.DataFrame(filas)[CARACTERISTICAS]))
    puntaje = pred.copy(); puntaje[0] *= (1 - UMBRAL)
    i = int(np.argmin(puntaje))
    CACHE_V3[clave] = (float(pred[i]), cands[i])
    return CACHE_V3[clave]

from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr
con_indice = pd.DataFrame(planes_con_indice).fillna(0).drop_duplicates(["consulta_id", "indice"]).set_index(["consulta_id", "indice"])
def opiniones(df, m_plan, m_idx):
    df = df[[(q, k) in con_indice.index for q, k in zip(df.consulta_id, df.indice)]].copy()
    sin = pd.DataFrame([extraer_caracteristicas(base_sin[q]["plan"], len(CONSULTAS[q][0])) for q in df.consulta_id])
    con = con_indice.loc[list(zip(df.consulta_id, df.indice))]
    t_sin = np.expm1(m_plan.predict(sin.reindex(columns=CARACTERISTICAS, fill_value=0)))
    t_con = np.expm1(m_plan.predict(con[CARACTERISTICAS]))
    df["op_planes"] = 1 - t_con / np.maximum(t_sin, 1e-6)
    df["op_indices"] = m_idx.predict(df[CARACT_INDICE])
    return df
op_val = opiniones(pares[pares.consulta_id.isin(ids_valida)], m_v3_val, m_idx_val)
apilado = LinearRegression(fit_intercept=False, positive=True).fit(op_val[["op_planes", "op_indices"]], op_val.mejora)
PESO_PLAN, PESO_INDICE = map(float, apilado.coef_)
if PESO_PLAN + PESO_INDICE == 0:
    PESO_PLAN = 1.0
op_pru = opiniones(pares_pru, modelo_v3, modelo_indices)
filas_ap = []
for nombre, df in [("validación", op_val), ("prueba (no vista)", op_pru)]:
    mezcla = PESO_PLAN * df.op_planes + PESO_INDICE * df.op_indices
    for quien, pred in [("solo modelo de planes", df.op_planes), ("solo modelo de índices", df.op_indices),
                        ("combinación aprendida", mezcla)]:
        filas_ap.append(dict(datos=nombre, predictor=quien, MAE=mean_absolute_error(df.mejora, pred),
                             spearman=spearmanr(df.mejora, pred).correlation))
print(f"\nPesos aprendidos: modelo de planes {PESO_PLAN:.2f} | modelo de índices {PESO_INDICE:.2f}")
print("¿Qué tan bien predice la mejora real de cada índice? (spearman: 1 = ordena perfecto)")
display(pd.DataFrame(filas_ap).style.format({"MAE": "{:.3f}", "spearman": "{:.2f}"}).hide(axis="index"))

class FormulaCostos:
    def predict(self, X):
        return np.log1p(np.asarray(X["raiz_costo"], dtype=float))
formula = FormulaCostos()

def tiempo_combinado(cur_g, q, indices, m_plan, mejoras_idx, pesos):
    t0 = mejor_plan(cur_g, q, [], m_plan)[0]
    m_p = 1 - mejor_plan(cur_g, q, indices, m_plan)[0] / max(t0, 1e-6)
    m_i = max([0.0] + [mejoras_idx.get((q, k), 0.0) for k in indices])
    return t0 * (1 - float(np.clip(pesos[0] * m_p + pesos[1] * m_i, -1, 1)))

def busqueda_conjunta(base, cur_g, consultas_q, m_plan, m_idx, corte, pesos=None, limite=None):
    pesos = pesos or (PESO_PLAN, PESO_INDICE)
    mejoras_idx = mejoras_predichas(base, consultas_q, m_idx)
    candidatos = sorted({k for _, k in mejoras_idx})
    if limite is None:
        limite = PRESUPUESTO * sum(mb_estimado(base, *k) for k in candidatos)
    def total(indices):
        return sum(tiempo_combinado(cur_g, q, indices, m_plan, mejoras_idx, pesos) for q in consultas_q)
    elegidos, usado = [], 0.0
    t_actual = t_inicio = total([])
    while True:
        opciones = []
        for k in candidatos:
            mb = mb_estimado(base, *k)
            if k in elegidos or usado + mb > limite:
                continue
            ganancia = t_actual - total(elegidos + [k])
            opciones.append((ganancia / mb, ganancia, k))
        opciones = [o for o in opciones if o[1] > max(corte * t_actual, 0)]
        if not opciones:
            break
        _, ganancia, k = max(opciones)
        elegidos.append(k); usado += mb_estimado(base, *k); t_actual -= ganancia
    return dict(indices=elegidos, mb=usado, limite=limite, pred_inicio=t_inicio, pred_final=t_actual, candidatos=candidatos)

prueba_corte = []
for corte in [0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]:
    filas = []
    for base, consultas_val in grupos_valida:
        if base not in GEMELAS or not consultas_val:
            continue
        usar_base(base)
        with GEMELAS[base].cursor() as cur_g:
            r = busqueda_conjunta(base, cur_g, consultas_val, m_v3_val, m_idx_val, corte)
        filas.append(dict(ganado=ganancia_real(consultas_val, r["indices"]),
                          ganado_todos=ganancia_real(consultas_val, r["candidatos"]),
                          mb=r["mb"], mb_todos=sum(mb_estimado(base, *k) for k in r["candidatos"])))
    puntaje, g, e = puntaje_equilibrio(filas)
    prueba_corte.append(dict(corte=corte, mejora_conseguida=g, espacio_usado=e, puntaje=puntaje))
prueba_corte = pd.DataFrame(prueba_corte)
CORTE = float(prueba_corte[prueba_corte.puntaje.round(3) == prueba_corte.puntaje.round(3).max()].corte.max())
print("\nCorte aprendido en validación:")
display(prueba_corte.style.format({"corte": "{:.1%}", "mejora_conseguida": "{:.0%}",
                                   "espacio_usado": "{:.0%}", "puntaje": "{:+.3f}"}).hide(axis="index"))
print(f"Corte elegido: un índice entra solo si baja el tiempo total predicho más de {CORTE:.1%}")

V3 = {}
for hechas, (base, consultas_prueba) in enumerate(grupos_prueba, 1):
    usar_base(base)
    with GEMELAS[base].cursor() as cur_g:
        r = busqueda_conjunta(base, cur_g, consultas_prueba, modelo_v3, modelo_indices, CORTE)
        control = busqueda_conjunta(base, cur_g, consultas_prueba, v3_barajado, falso, CORTE)
        solo_formula = busqueda_conjunta(base, cur_g, consultas_prueba, formula, modelo_indices, 0, pesos=(1.0, 0.0),
                                         limite=r["mb"] + 1e-9)
        elegidos = r["indices"]
        V3[base] = dict(indices=elegidos, mb=r["mb"], pred_inicio=r["pred_inicio"], pred_final=r["pred_final"],
                        planes={q: mejor_plan(cur_g, q, elegidos, modelo_v3)[1] for q in consultas_prueba},
                        planes_v1={q: mejor_plan(cur_g, q, [], modelo_v3)[1] for q in consultas_prueba},
                        planes_v2={q: mejor_plan(cur_g, q, RECOMENDADOS[base], modelo_v3)[1] for q in consultas_prueba},
                        control_indices=control["indices"],
                        control_planes={q: mejor_plan(cur_g, q, control["indices"], v3_barajado)[1] for q in consultas_prueba},
                        formula_indices=solo_formula["indices"],
                        formula_planes={q: mejor_plan(cur_g, q, solo_formula["indices"], formula)[1] for q in consultas_prueba})
        cur_g.execute("SELECT hypopg_reset()")
    cambia = sum(p != (None, ()) for p in V3[base]["planes"].values())
    print(f"\n[{base}] índices elegidos por la versión 3 (~{r['mb']:.1f} MB, presupuesto {r['limite']:.1f} MB):")
    for t, c in elegidos:
        print(f"   CREATE INDEX ON {t}({c});")
    print(f"   además cambia el plan de {cambia} de {len(consultas_prueba)} consultas | "
          f"tiempo predicho: {r['pred_inicio']:,.0f} -> {r['pred_final']:,.0f} ms | "
          f"control barajado elige {len(control['indices'])} índices | "
          f"solo fórmula elige {len(solo_formula['indices'])}")
    avance(hechas, len(grupos_prueba), inicio, f"búsqueda {base:15s} lista")
print(f"\nEXPLAIN hechos en las gemelas: {CONTADOR['explain_gemela']:,} | "
      f"consultas enviadas al servidor en este bloque: {CONTADOR['consultas_servidor']}")
fin_bloque("Bloque 11.2 - búsqueda conjunta", inicio)

In [ ]:
# BLOQUE 11.3 — V3: crea los índices de verdad y mide las tres versiones
def ddl_de(indices):
    return [f"CREATE INDEX ON {t}({c})" for t, c in indices]

inicio = time.time()
filas_v3 = []
for hechas, (base, consultas_prueba) in enumerate(grupos_prueba, 1):
    usar_base(base)
    r = V3[base]
    opciones = [
        ("PostgreSQL solo (sin índices)",            [],                         None),
        ("V1: modelo elige el plan",                 [],                         r["planes_v1"]),
        ("V2: índices del modelo",                   RECOMENDADOS[base],         None),
        ("V2 + V1 por separado",                     RECOMENDADOS[base],         r["planes_v2"]),
        ("V3: índices y planes juntos",              r["indices"],               r["planes"]),
        ("fórmula de costos, mismo espacio (sin ML)", r["formula_indices"],       r["formula_planes"]),
        ("control: V3 con modelos barajados",        r["control_indices"],       r["control_planes"]),
        ("todos los candidatos",                     list(CANDIDATOS_BASE[base]), None),
    ]
    with conexion.cursor() as cur:
        for nombre, indices, planes in opciones:
            total, mb = medir_con(cur, ddl_de(indices), consultas_prueba, planes)
            filas_v3.append(dict(base=base, opción=nombre, índices=len(indices), espacio_mb=mb, tiempo_ms=total))
        medir_con(cur, INDICES_BASE, [])      # dejar la base como en el bloque 3
    avance(hechas, len(grupos_prueba), inicio, f"base {base:15s} verificada")

DETALLE_V3 = pd.DataFrame(filas_v3)
VERSION3 = DETALLE_V3.groupby("opción", sort=False)[["índices", "espacio_mb", "tiempo_ms"]].sum().reset_index()
VERSION3["mejora vs PG solo"] = 1 - VERSION3.tiempo_ms / VERSION3.tiempo_ms.iloc[0]
print("Verificación real, suma de todas las bases (consultas de prueba):")
display(VERSION3.style.format({"espacio_mb": "{:.1f}", "tiempo_ms": "{:,.0f}", "mejora vs PG solo": "{:+.1%}"})
        .hide(axis="index"))

tabla_bases = DETALLE_V3.pivot_table(index="base", columns="opción", values="tiempo_ms", sort=False)
display((1 - tabla_bases.div(tabla_bases.iloc[:, 0], axis=0)).style.format("{:+.1%}"))

fig, ax = plt.subplots(figsize=(10, 4))
colores = ["#9ca3af", "#2563eb", "#16a34a", "#0d9488", "#ea580c", "#facc15", "#a855f7", "#6b7280"]
ax.bar(VERSION3["opción"], VERSION3["tiempo_ms"], color=colores)
for i, (t, mb) in enumerate(zip(VERSION3.tiempo_ms, VERSION3.espacio_mb)):
    ax.text(i, t, f"{mb:.0f} MB", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("tiempo total real (ms)"); ax.set_title("Versiones 1, 2 y 3 sobre las consultas de prueba")
ax.tick_params(axis="x", rotation=20); plt.tight_layout(); plt.show()
fin_bloque("Bloque 11.3 - verificación versión 3", inicio)

In [ ]:
# BLOQUE 12 — Comparación: búsqueda heurística que sí ejecuta las consultas
def aplicar_estado(cur, estado):
    orden, hash_, merge, nest = estado
    configurar_sesion(cur, forzar=orden is not None)
    cur.execute(f"SET enable_hashjoin={'on' if hash_ else 'off'}; "
                f"SET enable_mergejoin={'on' if merge else 'off'}; "
                f"SET enable_nestloop={'on' if nest else 'off'};")

def restaurar_sesion(cur):
    cur.execute("RESET enable_hashjoin; RESET enable_mergejoin; RESET enable_nestloop;")
    configurar_sesion(cur, forzar=False)

def vecinos(estado, tablas):
    orden, h, m, n = estado
    salida = []
    if orden is None:
        salida.append((tuple(tablas), h, m, n))           # forzar el orden escrito
    else:
        salida.append((None, h, m, n))                     # devolverle el orden a PostgreSQL
        for i in range(len(orden) - 1):                    # cambiar dos tablas vecinas de lugar
            nuevo = list(orden); nuevo[i], nuevo[i+1] = nuevo[i+1], nuevo[i]
            if orden_valido(nuevo):
                salida.append((tuple(nuevo), h, m, n))
    for j in range(3):                                     # prender/apagar cada método de JOIN
        metodos = [h, m, n]; metodos[j] = not metodos[j]
        if any(metodos):                                   # al menos uno debe quedar prendido
            salida.append((orden, *metodos))
    return salida

def escalada(cur, tablas, filtros):
    def medir(estado):
        aplicar_estado(cur, estado)
        try:
            return tiempo_real(cur, construir_sql(estado[0] or tablas, filtros))
        except Exception:
            return float("inf")
    actual = (None, True, True, True)
    t_actual = t_inicial = medir(actual)
    pruebas = 1
    while True:
        candidatos = [(medir(v), v) for v in vecinos(actual, tablas)]
        pruebas += len(candidatos)
        t_mejor, v_mejor = min(candidatos, key=lambda c: c[0])
        if t_mejor < t_actual * 0.95:
            actual, t_actual = v_mejor, t_mejor
        else:
            break
    restaurar_sesion(cur)
    return t_inicial, t_actual, pruebas, actual

inicio, hechas = time.time(), 0
heuristica = []
consultas_heur = sorted(prueba.consulta_id.unique())
for base, lista in por_base(consultas_heur):
    usar_base(base)
    with conexion.cursor() as cur:
        for q in lista:
            tablas, filtros = CONSULTAS[q]
            t_ini, t_fin, n_pruebas, estado = escalada(cur, tablas, filtros)
            hechas += 1; avance(hechas, len(consultas_heur), inicio, "consultas buscadas")
            heuristica.append(dict(consulta_id=q, base=base, inicial_ms=t_ini, final_ms=t_fin, pruebas=n_pruebas,
                                   orden="PostgreSQL" if estado[0] is None else " > ".join(estado[0]),
                                   metodos_apagados=", ".join(nm for nm, on in zip(["hash", "merge", "nest"], estado[1:]) if not on) or "-"))
heuristica = pd.DataFrame(heuristica)
HEUR_BASE, HEUR_FINAL = heuristica["inicial_ms"].sum(), heuristica["final_ms"].sum()
print(f"Búsqueda heurística sobre {len(heuristica)} consultas de prueba")
print(f"  PostgreSQL solo: {HEUR_BASE:,.0f} ms -> con la búsqueda: {HEUR_FINAL:,.0f} ms "
      f"({1 - HEUR_FINAL / HEUR_BASE:+.1%})")
print(f"  Ejecuciones reales necesarias: {heuristica['pruebas'].sum() * (CALENTAMIENTO + REPETICIONES):,} "
      f"(el modelo necesita 0: solo usa EXPLAIN)")
print(heuristica.assign(mejora=1 - heuristica.final_ms / heuristica.inicial_ms)
      .sort_values("mejora", ascending=False).head(8).round(2).to_string(index=False))
fin_bloque("Bloque 12 - búsqueda heurística", inicio)

In [ ]:
# BLOQUE 13 — Resultado final: PostgreSQL contra V1, V2 y V3
# Mismas consultas de prueba, todas partiendo de la base sin índices extra.
v3 = VERSION3.set_index("opción")
filas = [("PostgreSQL solo (consulta original)", "PostgreSQL solo (sin índices)"),
         ("V1: el modelo elige el plan",         "V1: modelo elige el plan"),
         ("V2: el modelo elige índices",         "V2: índices del modelo"),
         ("V3: índices y planes juntos",         "V3: índices y planes juntos")]
FINAL_TABLA = pd.DataFrame([dict(versión=nombre, índices=int(v3.loc[clave, "índices"]),
                                 espacio_mb=v3.loc[clave, "espacio_mb"], tiempo_ms=v3.loc[clave, "tiempo_ms"])
                            for nombre, clave in filas])
FINAL_TABLA["mejora"] = 1 - FINAL_TABLA.tiempo_ms / FINAL_TABLA.tiempo_ms.iloc[0]
display(FINAL_TABLA.style.format({"espacio_mb": "{:.1f}", "tiempo_ms": "{:,.0f}", "mejora": "{:+.1%}"})
        .hide(axis="index"))

versiones = FINAL_TABLA.iloc[1:]
mejor = versiones.loc[versiones.mejora.idxmax()]
original = FINAL_TABLA.tiempo_ms.iloc[0]
print(f"\nLa mejor es {mejor.versión}: las consultas pasan de {original:,.0f} ms a {mejor.tiempo_ms:,.0f} ms, "
      f"un {mejor.mejora:.1%} más rápidas que la consulta original.")
con_indices = versiones[versiones.espacio_mb > 0]
eficiente = con_indices.loc[(con_indices.mejora / con_indices.espacio_mb).idxmax()]
if eficiente.versión != mejor.versión:
    print(f"La que más mejora por cada MB de índices es {eficiente.versión}: "
          f"{eficiente.mejora:.1%} usando solo {eficiente.espacio_mb:.1f} MB.")

colores = ["#9ca3af"] * len(FINAL_TABLA)
colores[int(versiones.mejora.idxmax())] = "#ea580c"
fig, ax = plt.subplots(figsize=(9, 4))
barras = ax.barh(FINAL_TABLA.versión, FINAL_TABLA.tiempo_ms, color=colores)
for barra, m, mb in zip(barras, FINAL_TABLA.mejora, FINAL_TABLA.espacio_mb):
    texto = "original" if m == 0 else f"{m:+.1%}  ({mb:.0f} MB)"
    ax.text(barra.get_width(), barra.get_y() + barra.get_height() / 2, "  " + texto, va="center", fontsize=10)
ax.invert_yaxis()
ax.set_xlabel("tiempo total de las consultas de prueba (ms) — menos es mejor")
ax.set_title("¿Cuál optimiza mejor? (en naranjo, la mejor)")
ax.set_xlim(0, original * 1.3)
plt.tight_layout(); plt.show()

# Pruebas de que decide el Machine Learning y no solo un cálculo
print("\nMismo cálculo, sin lo aprendido:")
print(f"  V2 con el modelo barajado:               {VERSION2.set_index('opción').loc['control: modelo barajado', 'mejora vs sin índices']:+.1%}")
print(f"  V3 con la fórmula de PostgreSQL (sin ML): {v3.loc['fórmula de costos, mismo espacio (sin ML)', 'mejora vs PG solo']:+.1%}")
print(f"  V3 con los modelos barajados:            {v3.loc['control: V3 con modelos barajados', 'mejora vs PG solo']:+.1%}")
print(f"Aprendido en validación: presupuesto {PRESUPUESTO:.0%}, corte {CORTE:.0%}, "
      f"pesos de los modelos {PESO_PLAN:.2f} (planes) y {PESO_INDICE:.2f} (índices)")

print("\nTiempo de cada bloque:")
tiempos = pd.Series(TIEMPOS) / 60
print(tiempos.round(1).astype(str).add(" min").to_string())
print(f"Total: {tiempos.sum():.0f} min")

In [ ]:
# BLOQUE 14 — Lo que hace el modelo que una persona no: decidir sin ejecutar y descubrir sus propias reglas
from sklearn.tree import DecisionTreeClassifier, export_text

# Parte 1: una persona que solo cambia el JOIN tiene que probar planes ejecutándolos
grupos_v1 = {q: g for q, g in todas.groupby("consulta_id") if q in t_modelo.index}
PRUEBAS_AZAR = 30

def persona_prueba(k, como):
    total, gastado = 0.0, 0.0
    for q, g in grupos_v1.items():
        otros = g[g.orden != NATIVO]
        if como == "al azar":
            elegidos = [otros.sample(min(k, len(otros)), random_state=s) for s in range(PRUEBAS_AZAR)]
        else:
            elegidos = [otros.sort_values("raiz_costo").head(k)]
        total += np.mean([min(tiempos_nativos[q], e.tiempo_ms.min() if len(e) else np.inf) for e in elegidos])
        gastado += np.mean([e.tiempo_ms.sum() for e in elegidos])
    return total, gastado

MAX_K = int(max(len(g) for g in grupos_v1.values()) - 1)
filas_persona = []
for como in ["al azar", "los más baratos según PostgreSQL"]:
    for k in range(0, MAX_K + 1):
        total, gastado = persona_prueba(k, como)
        filas_persona.append(dict(persona=como, planes_probados=k, tiempo_ms=total, ms_gastados_probando=gastado))
PERSONA = pd.DataFrame(filas_persona)
PERSONA["mejora"] = 1 - PERSONA.tiempo_ms / TOTALES["nativo"]
MEJORA_MODELO_V1 = 1 - TOTALES["modelo"] / TOTALES["nativo"]

print(f"Solo JOIN, {len(grupos_v1)} consultas. El modelo no ejecuta nada y mejora {MEJORA_MODELO_V1:+.1%}: "
      f"ahorra {TOTALES['nativo'] - TOTALES['modelo']:,.0f} ms (el máximo posible es {1 - TOTALES['mejor'] / TOTALES['nativo']:+.1%}).")
for como, g in PERSONA.groupby("persona", sort=False):
    iguala = g[g.mejora >= MEJORA_MODELO_V1]
    if len(iguala):
        k = int(iguala.planes_probados.iloc[0])
        ahorro_p = TOTALES["nativo"] - iguala.tiempo_ms.iloc[0]
        print(f"  Persona que prueba {como}: necesita ejecutar {k} plan(es) por consulta para igualarlo. "
              f"Ahorra {ahorro_p:,.0f} ms, pero gasta {iguala.ms_gastados_probando.iloc[0]:,.0f} ms probando")
    else:
        print(f"  Persona que prueba {como}: no lo iguala ni probando todos los planes")

# Parte 2: reglas para crear índices, las de una persona contra las que descubre el modelo
NOMBRES = {"log_filas_tabla": "log10(filas de la tabla)", "es_filtro": "columna de filtro",
           "es_join": "columna de JOIN", "selectividad": "fracción de filas que quedan",
           "lee_entera": "hoy la lee entera", "parte_del_costo": "parte del costo de la consulta",
           "distintos": "valores distintos", "correlacion": "orden en disco",
           "n_tablas": "tablas en la consulta", "es_tabla_mas_grande": "es la tabla más grande"}

imita = (modelo_indices.predict(pares_des[CARACT_INDICE]) > .1).astype(int)
arbol = DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=SEMILLA)
arbol.fit(pares_des[CARACT_INDICE], imita)
def reglas_para_crear(arbol):
    t, salida = arbol.tree_, []
    def bajar(nodo, limites):
        if t.children_left[nodo] == -1:
            if t.value[nodo][0].argmax() == 1:
                texto = []
                for nombre, (bajo, alto) in limites.items():
                    if bajo is not None and alto is not None: texto.append(f"{nombre} entre {bajo:.2f} y {alto:.2f}")
                    elif bajo is not None:                    texto.append(f"{nombre} > {bajo:.2f}")
                    else:                                     texto.append(f"{nombre} <= {alto:.2f}")
                salida.append((texto, int(t.n_node_samples[nodo])))
            return
        nombre, corte = NOMBRES[CARACT_INDICE[t.feature[nodo]]], t.threshold[nodo]
        bajo, alto = limites.get(nombre, (None, None))
        bajar(t.children_left[nodo], {**limites, nombre: (bajo, corte)})
        bajar(t.children_right[nodo], {**limites, nombre: (corte, alto)})
    bajar(0, {})
    return salida

print("\nReglas que descubrió el modelo para CREAR un índice (nadie se las dio, salen de los tiempos medidos):")
for i, (condiciones, casos) in enumerate(reglas_para_crear(arbol), 1):
    print(f"  Regla {i}: SI " + " Y ".join(condiciones) + f"  ->  crear índice   ({casos} casos de entrenamiento)")
print("  En cualquier otro caso: no crear el índice")

REGLAS = {
    "persona: toda columna de JOIN":       (pares_pru.es_join == 1).values,
    "persona: toda columna de filtro":     (pares_pru.es_filtro == 1).values,
    "persona: si la tabla es grande":      (pares_pru.log_filas_tabla >= 4).values,
    "reglas que descubrió el modelo":      arbol.predict(pares_pru[CARACT_INDICE]).astype(bool),
    "modelo completo":                     modelo_indices.predict(pares_pru[CARACT_INDICE]) > .1,
}
sirve = (pares_pru.mejora > .1).values
ahorro = (pares_pru.t_sin * pares_pru.mejora).values
COMPARA_REGLAS = pd.DataFrame([dict(regla=n, índices_creados=int(r.sum()),
                                    inútiles=int((r & ~sirve).sum()),
                                    útiles_encontrados=(r & sirve).sum() / max(sirve.sum(), 1),
                                    acierto=(r == sirve).mean(),
                                    ms_ahorrados=ahorro[r].sum()) for n, r in REGLAS.items()])
print(f"Consultas de prueba: {pares_pru.consulta_id.nunique()} | pares (consulta, índice): {len(pares_pru)} | "
      f"índices que de verdad ayudan más de 10%: {sirve.sum()}")
display(COMPARA_REGLAS.style.format({"útiles_encontrados": "{:.0%}", "acierto": "{:.0%}",
                                     "ms_ahorrados": "{:,.0f}"}).hide(axis="index"))
print(f"Las reglas del árbol coinciden con el modelo completo en el "
      f"{(REGLAS['reglas que descubrió el modelo'] == REGLAS['modelo completo']).mean():.0%} de los casos de prueba")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for (como, g), color in zip(PERSONA.groupby("persona", sort=False), ["#9ca3af", "#16a34a"]):
    ax[0].plot(g.planes_probados, g.mejora * 100, "o-", color=color, label=f"persona: {como}")
ax[0].axhline(MEJORA_MODELO_V1 * 100, color="#ea580c", lw=2.5, label="modelo (0 ejecuciones)")
ax[0].set_xlabel("planes que la persona ejecuta por consulta"); ax[0].set_ylabel("mejora vs PostgreSQL (%)")
uno = PERSONA[(PERSONA.persona == "al azar") & (PERSONA.planes_probados == 1)].iloc[0]
ax[0].annotate(f"probar 1 plan por consulta:\nahorra {TOTALES['nativo'] - uno.tiempo_ms:,.0f} ms, "
               f"gasta {uno.ms_gastados_probando:,.0f} ms", (1, uno.mejora * 100), (3.5, 1),
               arrowprops=dict(arrowstyle="->"), fontsize=9)
ax[0].set_title("Solo JOIN: cuánto hay que ejecutar para igualar al modelo"); ax[0].legend(loc="upper left")
colores = ["#9ca3af"] * 3 + ["#2563eb", "#ea580c"]
ax[1].barh(COMPARA_REGLAS.regla[::-1], COMPARA_REGLAS.inútiles[::-1], color=colores[::-1])
for i, (u, c) in enumerate(zip(COMPARA_REGLAS.inútiles[::-1], COMPARA_REGLAS.índices_creados[::-1])):
    ax[1].text(u, i, f" {u} de {c}", va="center")
ax[1].set_xlabel("índices inútiles creados (ayudan menos de 10%)"); ax[1].set_title("Índices: reglas de persona vs del modelo")
plt.tight_layout(); plt.show()

In [ ]:
# BLOQUE 15 — Guarda los modelos de la V2 y la V3 para usarlos en la página
# Se copian a la carpeta ml/ del repo: modelo_v3.pkl (tiempo de un plan),
# modelo_indices.pkl (mejora de un índice) y parametros_v2_v3.json.
import joblib
joblib.dump(modelo_v3, "modelo_v3.pkl")
joblib.dump(modelo_indices, "modelo_indices.pkl")
PARAMETROS_APP = dict(caracteristicas=CARACTERISTICAS, caracteristicas_indice=CARACT_INDICE,
                      umbral=UMBRAL, presupuesto=PRESUPUESTO, corte=CORTE,
                      peso_plan=PESO_PLAN, peso_indice=PESO_INDICE)
with open("parametros_v2_v3.json", "w", encoding="utf-8") as f:
    json.dump(PARAMETROS_APP, f, ensure_ascii=False, indent=2)
print(json.dumps(PARAMETROS_APP, ensure_ascii=False, indent=2))
try:
    from google.colab import files
    for nombre in ("modelo_v3.pkl", "modelo_indices.pkl", "parametros_v2_v3.json"):
        files.download(nombre)
except ImportError:
    print("Archivos guardados en", os.getcwd())